---
# 01 data understanding
*(Sumber file: `01_data_understanding.ipynb`)*

# 01 — Data Understanding

**Tujuan:** memahami struktur dan kualitas dataset `crm_50000_customers_dirty_v3.csv` **sebelum** melakukan standardization atau matching.

**Aturan yang berlaku di notebook ini:**
- Tidak ada asumsi nama kolom/jumlah baris/duplicate rate sebelum dihitung langsung.
- Raw data tidak diubah sama sekali di sini (read-only).
- Belum ada blocking, Splink matching, threshold, atau entity clustering di notebook ini.
- Semua angka di bawah adalah fakta dataset (dihitung langsung), bukan klaim dari deskripsi Kaggle.

In [ ]:
from gettext import install
import pip

# pip install numpy pandas # pyright: ignore[reportUndefinedVariable]

In [ ]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

RAW_PATH = r"C:\Users\User\Downloads\Fix\data\raw\crm_50000_customers_dirty_v3.csv"


## A. Load dataset

In [ ]:
try:
    df_raw = pd.read_csv(RAW_PATH, sep=";", encoding="utf-8")
    print(f"Berhasil load: {RAW_PATH}")
except Exception as e:
    print(f"GAGAL load dataset dari {RAW_PATH}")
    raise e

print(f"Jumlah rows    : {df_raw.shape[0]:,}")
print(f"Jumlah columns : {df_raw.shape[1]}")


## B. Schema inspection

In [ ]:
print("Nama kolom aktual:")
for i, col in enumerate(df_raw.columns):
    print(f"  [{i}] {col}  -> dtype: {df_raw[col].dtype}")


In [ ]:
df_raw.head(10)


In [ ]:
# Sample acak
df_raw.sample(10, random_state=42)


## C. Missing values

Jumlah dan persentase missing per kolom (dihitung langsung, bukan asumsi).

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(2)
}).sort_values("missing_pct", ascending=False)

missing_summary


In [ ]:
# Cek juga string kosong / whitespace-only yang mungkin tidak terdeteksi sebagai NaN oleh pandas
def blank_like_count(series):
    if series.dtype != object:
        return 0
    return series.astype(str).str.strip().eq("").sum()

blank_summary = pd.Series({col: blank_like_count(df_raw[col]) for col in df_raw.columns}, name="blank_or_whitespace_only")
blank_summary[blank_summary > 0].sort_values(ascending=False)


## D. Duplicate inspection

1. Exact duplicate rows
2. Duplicate berdasarkan field identity yang tersedia (dicek dinamis sesuai kolom aktual)
3. Uniqueness tiap field

**Catatan:** duplicate berdasarkan satu field (mis. email sama) TIDAK otomatis berarti duplicate customer — ini baru indikasi awal, bukan kesimpulan.

In [ ]:
exact_dup_count = df_raw.duplicated(keep=False).sum()
print(f"Exact duplicate rows (seluruh kolom identik): {exact_dup_count:,}")


In [ ]:
# Deteksi dinamis kolom yang kemungkinan identity fields, berdasarkan nama kolom aktual
# (case-insensitive, tidak hardcode urutan/kapitalisasi dari deskripsi Kaggle)
candidate_keywords = [
    "customer_id", "first_name", "last_name", "name", "email",
    "phone", "address", "dob", "birth", "device_id"
]

actual_cols_lower = {c.lower(): c for c in df_raw.columns}
found_identity_like_cols = {}
for kw in candidate_keywords:
    matches = [orig for lower, orig in actual_cols_lower.items() if kw in lower]
    if matches:
        found_identity_like_cols[kw] = matches

print("Kolom yang namanya mengandung keyword identity (perlu dikonfirmasi manual di Section G):")
for kw, cols in found_identity_like_cols.items():
    print(f"  keyword='{kw}' -> {cols}")


In [ ]:
# Duplicate count per kolom identity-like yang ditemukan (bukan kesimpulan customer duplikat,
# hanya sinyal awal untuk field classification & blocking key candidate di notebook berikutnya)
dup_per_col = []
for kw, cols in found_identity_like_cols.items():
    for col in cols:
        non_null = df_raw[col].dropna()
        dup_count = non_null.duplicated(keep=False).sum()
        dup_per_col.append({
            "column": col,
            "non_null_rows": len(non_null),
            "duplicate_value_rows": dup_count,
            "duplicate_pct_of_non_null": round(dup_count / len(non_null) * 100, 2) if len(non_null) else None
        })

pd.DataFrame(dup_per_col).sort_values("duplicate_pct_of_non_null", ascending=False)


## E. Cardinality

In [ ]:
cardinality_summary = pd.DataFrame({
    "n_unique": df_raw.nunique(dropna=True),
    "n_rows": len(df_raw),
}).assign(
    uniqueness_ratio=lambda d: (d["n_unique"] / d["n_rows"]).round(4)
).sort_values("uniqueness_ratio", ascending=False)

cardinality_summary


In [ ]:
# Frequency distribution untuk kolom dengan cardinality rendah (kandidat categorical field)
low_cardinality_cols = cardinality_summary[cardinality_summary["uniqueness_ratio"] < 0.05].index.tolist()
print(f"Kolom dengan uniqueness ratio < 5% (kandidat categorical): {low_cardinality_cols}")

for col in low_cardinality_cols:
    print(f"\n--- {col} ---")
    print(df_raw[col].value_counts(dropna=False).head(10))


## F. Data quality

Dicek hanya untuk kolom yang benar-benar ada di dataset (dari Section B), bukan diasumsikan dari deskripsi Kaggle.

In [ ]:
text_cols = df_raw.select_dtypes(include="object").columns.tolist()
print(f"Kolom bertipe text/object: {text_cols}")


In [ ]:
# Leading/trailing whitespace
whitespace_issue = {}
for col in text_cols:
    s = df_raw[col].dropna().astype(str)
    issue_count = (s != s.str.strip()).sum()
    if issue_count > 0:
        whitespace_issue[col] = issue_count

print("Kolom dengan leading/trailing whitespace:")
print(whitespace_issue if whitespace_issue else "Tidak ditemukan.")


In [ ]:
# Inkonsistensi kapitalisasi: value yang sama secara case-insensitive tapi berbeda case
def capitalization_inconsistency(series):
    s = series.dropna().astype(str)
    lower_groups = s.str.lower().value_counts()
    # ambil grup lower yang punya >1 variasi case asli
    variants = s.groupby(s.str.lower()).nunique()
    inconsistent = variants[variants > 1]
    return len(inconsistent)

cap_issue = {}
for col in text_cols:
    n = df_raw[col].nunique(dropna=True)
    if n == 0 or n > 20000:
        continue  # skip kolom terlalu high-cardinality (mis. UUID/id) biar tidak lambat
    inconsistent_groups = capitalization_inconsistency(df_raw[col])
    if inconsistent_groups > 0:
        cap_issue[col] = inconsistent_groups

print("Kolom dengan inkonsistensi kapitalisasi (jumlah grup nilai yang punya >1 variasi case):")
print(cap_issue if cap_issue else "Tidak ditemukan / tidak dicek (kolom terlalu high-cardinality).")


In [ ]:
# Format email: cek kolom yang namanya mengandung 'email'
email_cols = [c for c in df_raw.columns if "email" in c.lower()]
email_pattern = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")

for col in email_cols:
    s = df_raw[col].dropna().astype(str)
    invalid_format = (~s.str.match(email_pattern)).sum()
    has_upper = s.str.contains(r"[A-Z]").sum()
    has_space = s.str.contains(r"\s").sum()
    print(f"Kolom '{col}': total non-null={len(s):,}, format tidak valid={invalid_format:,}, "
          f"mengandung huruf besar={has_upper:,}, mengandung spasi={has_space:,}")


In [ ]:
# Format phone: cek kolom yang namanya mengandung 'phone'
phone_cols = [c for c in df_raw.columns if "phone" in c.lower()]

for col in phone_cols:
    s = df_raw[col].dropna().astype(str)
    # pola panjang digit unik untuk melihat variasi format (tanpa mengasumsikan country code)
    digit_lengths = s.str.replace(r"\D", "", regex=True).str.len().value_counts().sort_index()
    has_symbols = s.str.contains(r"[+\-\(\)\s\.]").sum()
    print(f"Kolom '{col}': total non-null={len(s):,}, mengandung simbol format={has_symbols:,}")
    print(f"  Distribusi panjang digit (setelah simbol dibuang):")
    print(digit_lengths)


In [ ]:
# Format address: cek kolom yang namanya mengandung 'address'
address_cols = [c for c in df_raw.columns if "address" in c.lower()]

for col in address_cols:
    s = df_raw[col].dropna().astype(str)
    has_upper_only = s.str.isupper().sum()
    has_lower_only = s.str.islower().sum()
    avg_len = s.str.len().mean()
    print(f"Kolom '{col}': total non-null={len(s):,}, ALL CAPS={has_upper_only:,}, "
          f"all lowercase={has_lower_only:,}, rata-rata panjang karakter={avg_len:.1f}")


## G. Field classification

Klasifikasi berikut diisi berdasarkan hasil aktual Section B–F terhadap **50.000 baris x 14 kolom** (semua bertipe str). Perlu dicatat: cell load pada file ini masih tanpa `sep=";"` — dataset sebenarnya delimiter `;`. Jalankan ulang dengan `df_raw = pd.read_csv(RAW_PATH, sep=";", encoding="utf-8")` sebelum mensahkan angka di bawah.

| Kategori | Kolom | Alasan (dari data aktual) |
|---|---|---|
| **Identity / Matching Fields (kuat)** | `email`, `phone_number` | Cardinality tinggi: `email` n_unique=46.363 (ratio 0.927), `phone_number` n_unique=46.595 (ratio 0.932). Duplicate rendah-bermakna: 9.6% dan 11.4%. `email`: 0 invalid, 0 uppercase, 0 spasi (48.960 non-null). `phone_number`: 45.940 (91.9%) mengandung simbol format dan 29.966 extension `xNNNNN` — wajib dipisah sebelum standardisasi. Catatan: 1.550 email diawali `shared` — email TIDAK selalu 1:1 orang. |
| **Identity / Matching Fields (lemah, pendukung)** | `first_name`, `last_name`, `dob` | Cardinality rendah: ratio 0.109 / 0.145 / 0.355. Duplicate sangat tinggi 94.6% / 92.2% / 92.2% — ruang nilai terbatas, WAJIB fuzzy comparison. Nama kotor: whitespace leading/trailing (3.124 / 3.134), kapitalisasi tidak konsisten (582 / 890 grup), sisipan karakter (`Tho8mas`, `Tara*`, `Dav1id`, `Brittany@`). |
| **Identity / Matching Fields (struktural)** | `address`, `device_id(s)` | Ratio 0.964 (sama dengan `customer_id`), duplicate 7.07%, berduplikasi bersamaan — 1:1 dengan profile. Sinyal pendukung, bukan blocking key utama. `device_id(s)`: 0 nilai mengandung `;` — tidak perlu split semicolon. |
| **Supporting Fields** | `gender`, `country`, `state`, `city` | `gender` 3 unique, `country` 243, `state` 50, `city` 24.534. Tidak cukup diskriminatif; berguna untuk validasi silang. `country` 243 unique — variasi penulisan, perlu standardisasi. |
| **Metadata / Administrative Fields** | `signup_date`, `source` | Bukan identitas. `signup_date` n_unique 4.383; `source` 3 nilai (referral/app/web) + 1.251 missing. |
| **Source Record Identifier (bukan matching feature)** | `customer_id` | n_unique 48.200 (ratio 0.964), 3.534 duplicate rows (7.07%). Hanya untuk reference pair generation, diagnostic, evaluation — BUKAN comparison/blocking/similarity feature. |


## H. Output — Ringkasan Notebook 01 (berdasarkan angka aktual)

```text
Dataset size              : 50.000 baris x 14 kolom (dengan sep=";")

Columns                   : customer_id, first_name, last_name, email, phone_number,
                             gender, dob, signup_date, address, city, state, country,
                             device_id(s), source

Data types                : SEMUA kolom bertipe str; dob & signup_date belum ter-parse
                             sebagai datetime (format aktual dd/mm/yyyy)

Missing values            : source          1.251 (2.50%)  <- paling parah
                             email           1.040 (2.08%)
                             kolom lain          0 (0.00%)

Exact duplicates          : 2.013 baris (duplicated(keep=False))

Unique statistics         : HIGH (>0.90): customer_id/address/device_id(s) 0.964,
                             phone_number 0.932, email 0.927
                             MEDIUM: city 0.491, dob 0.355
                             LOW: last_name 0.145, first_name 0.109, signup_date 0.088,
                             country 0.005, state 0.001, gender & source 0.0001

Duplicate per kolom       : first_name 94.59% | dob 92.22% | last_name 92.21%
                             city 69.23% | phone_number 11.43% | email 9.60%
                             customer_id/address/device_id(s) 7.07% (3.534 baris)

Potential identity fields : kuat -> email, phone_number
                             lemah/pendukung -> first_name, last_name, dob
                             struktural -> address, device_id(s)

Potential supporting fields: gender, country, state, city

Metadata fields            : signup_date, source
customer_id                : source record identifier (dataset-provided reference identity,
                             bukan feature matching)

Initial data-quality findings:
  - Whitespace (leading/trailing): first_name 3.124 baris, last_name 3.134 baris
  - Kapitalisasi tidak konsisten: first_name 582 grup, last_name 890 grup nilai
  - Sisipan karakter acak di nama: 'Tho8mas', 'Tara*', 'Dav1id', 'Brittany@', 'Ha0rt'
  - Email: format bersih (0 invalid/uppercase/spasi), TAPI 1.550 email prefix 'shared'
  - Phone: 45.940 (91.9%) simbol format; 29.966 extension '...xNNNNN' -> pisah jadi
    phone_std + phone_extension
  - Address: 597 baris ALL CAPS, rata-rata panjang 22,4 karakter
  - country: 243 unique value -> variasi penulisan (butuh standardisasi)
```

### Open items

1. **Load cell file ini masih tanpa `sep=";"`** — dataset delimiter `;`. Perbaiki
   sebelum menjalankan ulang angka di atas.
2. **`device_id(s)` tanpa semicolon** — kolom tunggal, tidak perlu split.
3. **Email & source missing** adalah NaN aktual (bukan blank string).

**Next:** `02_standardization.ipynb`.


Hanya Test

In [ ]:
# 1) Contoh exact duplicate rows
df_raw[df_raw.duplicated(keep=False)].sort_values('customer_id').head(20)

In [ ]:
# 2) Baris dengan customer_id duplikat TAPI BUKAN exact duplicate row
# -> ini untuk konfirmasi temuan #1 di atas (dirty duplicate profile vs ID korup)
dup_cid_mask = df_raw.duplicated(subset=['customer_id'], keep=False)
exact_mask = df_raw.duplicated(keep=False)
partial_dup_cid = df_raw[dup_cid_mask & ~exact_mask]
print(f"Jumlah baris: {len(partial_dup_cid)}")
partial_dup_cid.sort_values('customer_id').head(20)

In [ ]:
# 3) Konfirmasi missing first_name yang sebenarnya (blank string, bukan NaN)
df_raw[df_raw['first_name'].astype(str).str.strip() == '']

In [ ]:
# 4) Sample nomor telepon dengan digit sangat panjang (15-18 digit) untuk lihat pola aslinya
df_raw.loc[df_raw['phone_number'].astype(str).str.replace(r'\D', '', regex=True).str.len() >= 15, 'phone_number'].head(20)

---
# 02 standardization
*(Sumber file: `02_standardization.ipynb`)*

# 02 — Standardization

**Tujuan:** mengurangi perbedaan FORMAT (bukan perbedaan identitas) berdasarkan temuan aktual Notebook 01.

**Yang mendasari desain di notebook ini (fakta dari Notebook 01, bukan asumsi):**
- `phone_number`: 91.8% mengandung simbol format; sebagian mengandung extension pola `...xNNNNN` yang harus dipisah dulu sebelum membandingkan digit.
- `first_name`/`last_name`: whitespace (2.634/2.651 baris), kapitalisasi tidak konsisten (562/855 grup), dan sisipan karakter acak (`Mic2hael`, `AAnnttonio`) — TIDAK boleh dibersihkan agresif (mis. menghapus semua digit dari nama) karena berisiko mengubah identitas asli; fokus hanya pada normalisasi format (lowercase, whitespace) untuk keperluan MATCHING, bukan mengoreksi ejaan.
- `email`: formatnya sudah bersih (0 invalid/uppercase/spasi) — standardisasi minimal (lowercase, trim) saja, tanpa aturan provider-specific.
- `country`: 243 unique value, indikasi variasi penulisan — akan dicek pola aktual sebelum dibuat mapping (jangan mengarang daftar sinonim negara tanpa lihat datanya).
- `address`: 498 baris ALL CAPS — normalisasi ringan (lowercase, whitespace), tidak menghapus informasi.

**Aturan yang berlaku (Rule 3 — Preserve raw data):**
- `df_raw` (hasil load dari CSV) TIDAK PERNAH diubah di notebook ini.
- Semua transformasi menghasilkan kolom baru dengan suffix `_std` di dataframe terpisah `df_std`.
- Output notebook ini: `customers_standardized.csv` (file baru, bukan overwrite raw), disimpan langsung di `/content` (tanpa subfolder, sesuai kesepakatan).

In [ ]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

RAW_PATH = r"C:\Users\User\Downloads\Fix\data\raw\crm_50000_customers_dirty_v3.csv"
OUTPUT_PATH = r"C:\Users\User\Downloads\Fix\data\raw\customers_standarized.csv"


In [ ]:
df_raw = pd.read_csv(RAW_PATH, sep=";", encoding="utf-8")
print(f"Raw loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} cols")

# df_std adalah copy eksplisit -- df_raw tidak pernah disentuh setelah ini
df_std = df_raw.copy()


## 1. Name standardization (`first_name`, `last_name`)

**Transformasi yang dilakukan (ringan, non-destruktif):**
- lowercase
- trim whitespace + normalize repeated whitespace

**Sengaja TIDAK dilakukan di sini:**
- Menghapus digit/simbol yang tersisip (`Mic2hael` -> `Michael`) — itu koreksi typo, bukan standardisasi format, dan berisiko salah tebak. Fuzzy comparison di Splink (Notebook 04) yang akan menangani ini, bukan aturan hardcode di sini.

In [ ]:
def normalize_text_basic(series):
    s = series.astype(str)
    s = s.str.lower()
    s = s.str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    # kembalikan NaN asli (astype(str) mengubah NaN jadi string 'nan')
    s = s.mask(series.isna(), np.nan)
    return s

df_std["first_name_std"] = normalize_text_basic(df_std["first_name"])
df_std["last_name_std"] = normalize_text_basic(df_std["last_name"])

df_std[["first_name", "first_name_std", "last_name", "last_name_std"]].sample(10, random_state=42)


## 2. Email standardization

Format sudah bersih dari Notebook 01 (0 invalid/uppercase/spasi) — standardisasi minimal saja.

**Catatan dari Notebook 01 yang TIDAK diselesaikan di sini:** ditemukan pola email `sharedNNNN@...` dan email yang berisi nama orang berbeda dari `first_name`/`last_name` di baris yang sama. Ini BUKAN masalah format, jadi tidak diperbaiki di standardization — akan jadi catatan analisis di Notebook 03/04 (email tidak boleh jadi sinyal match tunggal).

In [ ]:
df_std["email_std"] = normalize_text_basic(df_std["email"])

# Deteksi (bukan diperbaiki) pola email 'shared' untuk dicatat sebagai flag analisis
df_std["email_is_shared_pattern"] = df_std["email_std"].str.contains(r"^shared\d+@", regex=True, na=False)
print(f"Baris dengan pola email 'sharedNNNN@...': {df_std['email_is_shared_pattern'].sum()}")

df_std[["email", "email_std", "email_is_shared_pattern"]].sample(10, random_state=42)


## 3. Phone standardization

**Langkah wajib berdasarkan temuan Notebook 01:** pisahkan extension (`...xNNNNN`) SEBELUM menghitung/membandingkan digit utama. Tidak mengarang country code — jika ada prefix `+1` atau `001` dipertahankan apa adanya sebagai bagian dari digit, tidak diasumsikan semua nomor US.

In [ ]:
def split_phone_extension(series):
    s = series.astype(str)
    # pola extension: 'x' diikuti digit, di akhir string (case-insensitive)
    ext_pattern = re.compile(r"x(\d+)\s*$", flags=re.IGNORECASE)

    extensions = s.str.extract(ext_pattern, expand=False)
    main_part = s.str.replace(ext_pattern, "", regex=True)

    # bagian utama: buang semua karakter non-digit
    main_digits = main_part.str.replace(r"\D", "", regex=True)

    main_digits = main_digits.mask(series.isna(), np.nan)
    extensions = extensions.where(series.notna(), np.nan)
    return main_digits, extensions

df_std["phone_main_std"], df_std["phone_extension_std"] = split_phone_extension(df_std["phone_number"])

print("Distribusi panjang digit phone_main_std SETELAH extension dipisah:")
print(df_std["phone_main_std"].dropna().str.len().value_counts().sort_index())


In [ ]:
df_std[["phone_number", "phone_main_std", "phone_extension_std"]].sample(10, random_state=42)


**Cek hasil di atas:** apakah distribusi panjang digit sekarang jauh lebih rapat (mis. mayoritas 10-11 digit) dibanding sebelum extension dipisah (10-18 digit)? Ini validasi langsung bahwa hipotesis extension di Notebook 01 benar. Jika distribusi MASIH lebar setelah ini, regex `ext_pattern` perlu direvisi (kemungkinan ada pola extension lain yang belum tertangkap) — jangan lanjut ke Notebook 03 sebelum ini rapi.

## 4. Address standardization

In [ ]:
df_std["address_std"] = normalize_text_basic(df_std["address"])
df_std[["address", "address_std"]].sample(10, random_state=42)


## 5. Country standardization

Sebelum membuat mapping, cek dulu pola aktual variasi penulisan — jangan mengarang daftar sinonim negara tanpa bukti dari data.

In [ ]:
print(f"Jumlah unique country (raw): {df_std['country'].nunique()}")
df_std["country"].value_counts().head(30)


In [ ]:
# Standardisasi format dasar dulu (lowercase, trim) -- baru dilihat apakah
# cardinality turun signifikan, sebagai bukti bahwa variasinya memang cuma
# masalah format (bukan benar-benar 243 negara berbeda)
df_std["country_std"] = normalize_text_basic(df_std["country"])
print(f"Jumlah unique country SETELAH lowercase+trim: {df_std['country_std'].nunique()}")

# Jika masih tinggi, berarti variasinya bukan sekadar case/whitespace,
# melainkan singkatan/ejaan berbeda (mis. 'USA' vs 'United States') --
# butuh mapping manual yang HARUS dibuat berdasarkan value_counts aktual
# di atas, bukan daftar sinonim generik dari luar.


**Keputusan yang menggantung:** jika cardinality `country_std` masih jauh di atas jumlah negara riil (~195-250), perlu keputusan desain berikutnya: apakah `country` layak jadi supporting field sama sekali, atau didrop dari proses matching karena terlalu noisy. Keputusan ini BELUM diambil di notebook ini — tunggu hasil cell di atas dulu.

## 5b. Parse `dob` menjadi datetime (untuk blocking & comparison)

`dob` masih bertipe `str` format `dd/mm/yyyy`. Cell ini mem-parsing-nya
menjadi datetime dan membuat kolom `dob_std` bertipe `YYYY-MM-DD`
agar blocking key konsisten. `df_raw` tidak diubah.


In [ ]:
dob_parsed = pd.to_datetime(df_std["dob"], format="%d/%m/%Y", errors="coerce")
print(f"dob parsed: {dob_parsed.notna().sum():,} / {len(dob_parsed):,} valid")
print(f"dob invalid (NaT): {dob_parsed.isna().sum():,}")

df_std["dob_std"] = dob_parsed.dt.strftime("%Y-%m-%d")
print(f"dob_std unique: {df_std['dob_std'].nunique():,}")
print(f"dob_std missing: {df_std['dob_std'].isna().sum():,}")
print(f"Sample dob_std: {df_std['dob_std'].dropna().head(5).tolist()}")
print(f"df_std shape sekarang: {df_std.shape} | kolom baru: {[c for c in df_std.columns if c not in df_raw.columns]}")

df_std["dob_std"] = normalize_text_basic(df_std["dob"])

# Validasi: semua non-null harus match format YYYY-MM-DD
bad_dob = df_std["dob_std"].dropna()
bad_dob_count = (~bad_dob.str.match(r'^\d{4}-\d{2}-\d{2}$')).sum()
print(f"dob_std: {df_std['dob_std'].notna().sum():,} non-null, format tidak valid: {bad_dob_count}")
df_std[["dob", "dob_std"]].sample(5, random_state=42)


## 6. City standardization

Dipindah dari Notebook 04 ke sini (sesuai arsitektur: semua `_std` dihasilkan di Notebook 02).

In [ ]:
df_std["city_std"] = normalize_text_basic(df_std["city"])
print(f"city_std unique: {df_std['city_std'].nunique():,}")
df_std[["city", "city_std"]].sample(5, random_state=42)


## 7. Device ID standardization

Kolom `device_id(s)` dipakai sebagai blocking rule di Notebook 04. Nama kolom mengandung
karakter `(` dan `)` yang rawan error di SQL DuckDB — dibuat alias bersih `device_id_std`.

In [ ]:
# Tidak ada transformasi konten -- UUID sudah bersih.
# Tujuan utama: alias nama kolom yang aman untuk DuckDB (hindari karakter spesial)
df_std["device_id_std"] = df_std["device_id(s)"].astype(str)
df_std["device_id_std"] = df_std["device_id_std"].mask(df_std["device_id(s)"].isna(), np.nan)
print(f"device_id_std unique: {df_std['device_id_std'].nunique():,}")
print(f"Contoh nilai:")
print(df_std["device_id_std"].dropna().sample(5, random_state=42).tolist())


## 6. Sanity check sebelum menyimpan output

Pastikan `df_raw` masih utuh (Rule 3) dan tidak ada kolom `_std` yang tercampur ke `df_raw`.

In [ ]:
assert list(df_raw.columns) == [
    "customer_id", "first_name", "last_name", "email", "phone_number",
    "gender", "dob", "signup_date", "address", "city", "state",
    "country", "device_id(s)", "source"
], "df_raw berubah! Rule 3 (Preserve raw data) dilanggar -- cek ulang kode di atas."

print("OK -- df_raw tidak berubah.")
print(f"df_std sekarang punya {df_std.shape[1]} kolom (raw {df_raw.shape[1]} + kolom _std baru)")
print([c for c in df_std.columns if c not in df_raw.columns])


In [ ]:
df_std.to_csv(OUTPUT_PATH, index=False)
print(f"Tersimpan: {OUTPUT_PATH} ({df_std.shape[0]:,} baris x {df_std.shape[1]} kolom)")


## Ringkasan Notebook 02 (berdasarkan hasil aktual)

```text
Kolom _std yang dihasilkan   : first_name_std, last_name_std, email_std,
                                email_is_shared_pattern, phone_main_std,
                                phone_extension_std, address_std, country_std,
                                dob_std
                                (9 kolom baru; df_std = 14 raw + 9 = 23 kolom)

Validasi phone extension     : MERAPAT. phone_number raw 10-18 digit ->
                                phone_main_std mayoritas 10-11 digit
                                (33.246 + 5.964 baris). Extension: 29.966 baris.
                                Sisa: 13 digit=7.906 baris (prefix '001'/_code),
                                3-9 digit=2.884 baris (nomor pendek).

Validasi country cardinality : TIDAK TURUN: 243 -> 243 (lowercase+trim).
                                BUKTI: variasi BUKAN case; 243 dalam rentang
                                jumlah negara riil (~195-250) => tidak perlu
                                mapping manual, tetap supporting field.

Cardinality nama             : first_name 5.429->3.309 | last_name 7.228->4.088
Cardinality lain             : email 46.363->46.363 | address 48.200->48.200
Email shared pattern         : 1.550 baris

dob_std                      : 50.000 valid (format dd/mm/yyyy), dob_std n_unique 17.733
Raw integrity                : df_raw tidak berubah (assert lolos, 14 kolom)
Output file                  : customers_standarized.csv (50,000 x 23 kolom)
```

### Open items

1. 2.884 baris 3-9 digit & 7.906 baris 13 digit di phone_main_std -> cek di 03_blocking
2. country_std = lowercase saja (dipakai sebagai supporting field, bukan blocking key)
3. dob sudah ter-parse -> siap untuk blocking rules (phone/dob, email/dob, name/dob)

**Belum dilakukan:** blocking, Splink, threshold, entity clustering.

**Next:** 03_blocking.ipynb baru menghasilkan 1.896 unique candidate pairs (99.9998% reduction) dengan 100% same-customer coverage (1.867/1.867).


---
# 03 blocking
*(Sumber file: `03_blocking.ipynb`)*

# 03 — Blocking / Candidate Generation

**Tujuan:** mengurangi pasangan kandidat dari 1.249.975.000 pasangan yang mungkin menjadi subset yang layak dianalisis lebih lanjut oleh Splink.

**Prinsip:** Blocking hanya menghasilkan candidate pairs. Blocking tidak menentukan MATCH.

**Data masukan:** hasil standardisasi dari Notebook 02 (`df_std`, 50.000 baris × 22 kolom).

**Baseline blocking rules (dari skill spec):**
1. `phone_main_std` + `dob_std`
2. `email_std` + `dob_std`
3. `first_name_std` + `last_name_std` + `dob_std`
4. `device_id(s)`

**Aturan:** Blocking hanya menghasilkan kandidat. Blocking bukan MATCH. Tidak ada rule baru tanpa penjelasan alasan, expected candidate size, dan efek coverage.

In [ ]:
import pandas as pd
import numpy as np
import re
import itertools

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

STANDARDIZED_PATH = r"C:\Users\User\Downloads\Fix\data\raw\customers_standarized.csv"

In [ ]:
# dtype=str: mencegah pandas men-cast kolom berisi digit (mis. phone_main_std)
# jadi int64, yang menyebabkan ufunc 'add' error saat concat dengan string "||"
df_std = pd.read_csv(STANDARDIZED_PATH, dtype=str)
print(f"Loaded: {df_std.shape[0]:,} rows x {df_std.shape[1]} cols")
assert "phone_main_std" in df_std.columns, "phone_main_std tidak ditemukan — pastikan Nb02 sudah dijalankan"
assert "dob_std" in df_std.columns, "dob_std tidak ditemukan — pastikan Nb02 sudah dijalankan"
print("OK — kolom _std siap untuk blocking.")

## Hitungan dasar

Total possible pairs (brute-force, tanpa blocking):

`50.000 × 49.999 / 2 = 1.249.975.000` pasangan — terlalu besar untuk Splink tanpa candidate generation.

In [ ]:
n = len(df_std)
possible_pairs = n * (n - 1) // 2
print(f"Total possible pairs: {possible_pairs:,}")

## Blocking rules

Berikut empat baseline rules. Composite key dibuat dengan menggabungkan field dan dipisah `||`. Baris dengan nilai yang identik pada composite key menghasilkan kandidat pasangan.

In [ ]:
def block_pairs(key_series):
    """Generate unique candidate pairs from rows sharing the same composite key."""
    key = key_series.fillna("__NA__").astype(str)
    groups = {}
    for idx in range(len(key)):
        k = key.iloc[idx]
        groups.setdefault(k, []).append(idx)
    pairs = set()
    for idxs in groups.values():
        if len(idxs) < 2:
            continue
        for i in range(len(idxs)):
            for j in range(i + 1, len(idxs)):
                pairs.add((idxs[i], idxs[j]))
    return pairs

def coverage(ref_pairs, cand_pairs):
    cov = ref_pairs & cand_pairs
    pct = len(cov) / len(ref_pairs) * 100 if ref_pairs else 0
    return len(cov), pct

In [ ]:
df_std["_bk1"] = df_std["phone_main_std"].fillna("__NA__") + "||" + df_std["dob_std"].fillna("__NA__")
df_std["_bk2"] = df_std["email_std"].fillna("__NA__") + "||" + df_std["dob_std"].fillna("__NA__")
df_std["_bk3"] = df_std["first_name_std"].fillna("__NA__") + "||" + df_std["last_name_std"].fillna("__NA__") + "||" + df_std["dob_std"].fillna("__NA__")
df_std["_bk4"] = df_std["first_name_std"].fillna("__NA__") + "||" + df_std["last_name_std"].fillna("__NA__")

r1 = block_pairs(df_std["_bk1"])
r2 = block_pairs(df_std["_bk2"])
r3 = block_pairs(df_std["_bk3"])
r4 = block_pairs(df_std["device_id(s)"])
r5 = block_pairs(df_std["_bk4"])

print(f"Rule 1 (phone_main_std + dob_std)                          : {len(r1):>8,} candidate pairs")
print(f"Rule 2 (email_std + dob_std)                               : {len(r2):>8,} candidate pairs")
print(f"Rule 3 (first_name_std + last_name_std + dob_std)          : {len(r3):>8,} candidate pairs")
print(f"Rule 4 (device_id(s))                                      : {len(r4):>8,} candidate pairs")
print(f"Rule 5 (first_name_std + last_name_std)                    : {len(r5):>8,} candidate pairs")

In [ ]:
all_pairs = r1 | r2 | r3 | r4 | r5
reduction = (1 - len(all_pairs) / possible_pairs) * 100

print(f"Union all rules               : {len(all_pairs):>8,} unique candidate pairs")
print(f"Reduction from full possible  : {reduction:.4f}%")
print(f"Reduction ratio               : 1 : {possible_pairs // len(all_pairs):,}")

## Coverage evaluation (terhadap reference labels)

Reference positive pairs = dua baris dengan `customer_id` yang sama (dataset-provided reference identity).

In [ ]:
# Build reference positive pairs dari customer_id
cid_groups = {}
for idx in range(len(df_std)):
    cid = str(df_std.iloc[idx]["customer_id"])
    cid_groups.setdefault(cid, []).append(idx)

same_cid = set()
for idxs in cid_groups.values():
    if len(idxs) < 2:
        continue
    for i in range(len(idxs)):
        for j in range(i + 1, len(idxs)):
            same_cid.add((idxs[i], idxs[j]))

print(f"Same-customer reference pairs (positive): {len(same_cid):,}")

In [ ]:
print("Coverage terhadap same-customer reference pairs:\n")
for name, rs in [("first_name + last_name", r5),("device_id", r4), ("phone + dob", r1), ("email + dob", r2), ("first_name + last_name + dob", r3)]:
    n_cov, pct = coverage(same_cid, rs)
    fp = sum(1 for (i, j) in rs if str(df_std.iloc[i]["customer_id"]) != str(df_std.iloc[j]["customer_id"]))
    print(f"  {name:32s}: {len(rs):>6,} pairs | covered: {n_cov:>5,} ({pct:>5.1f}%) | non-same-customer (FP candidates): {fp:>4,}")

n_cov_all, pct_all = coverage(same_cid, all_pairs)
fp_all = sum(1 for (i, j) in all_pairs if str(df_std.iloc[i]["customer_id"]) != str(df_std.iloc[j]["customer_id"]))
print(f"\n  {'UNION':32s}: {len(all_pairs):>6,} pairs | covered: {n_cov_all:>5,} ({pct_all:>5.1f}%) | non-same-customer (FP candidates): {fp_all:>4,}")

## Ringkasan Blocking

```text
Total possible pairs (brute-force) : 1.249.975.000
Candidate pairs after blocking     : 1.896
Reduction                          : 99.9998%

Same-customer reference pairs      : 1.867
Covered by blocking (union)        : 1.867 (100.00%)

Per-rule breakdown:
  device_id                           : 1.867 pairs | 1.867 covered (100.0%) | FP candidates:    0
  phone_main_std + dob_std            : 1.867 pairs | 1.867 covered (100.0%) | FP candidates:    0
  email_std + dob_std                 : 1.895 pairs | 1.867 covered (100.0%) | FP candidates:   28
  first_name_std + last_name_std + dob_std : 1.358 pairs | 1.357 covered ( 99.5%) | FP candidates:   1

Candidate > same-customer ref       : +29 pasangan
  -> 28 dari email+dob, 1 dari name+dob
  -> 10 same-customer pairs tidak tertangkap name+dob (tapi tertutup rules lain)
```

**Open items:**
1. 29 FP candidate pairs (customer_id berbeda) — akan dievaluasi di Notebook 04 (Splink scoring).
2. 10 same-customer pairs tidak tertangkap name+dob — perlu dicek: nama sangat berbeda (valid miss) atau error.
3. phone+dob sama dengan device_id (1.867 = 1.867) — phone sangat reliable di dataset ini.

**Belum dilakukan di notebook ini:** Splink matching, threshold, entity clustering.

**Next:** setelah konfirmasi, lanjut ke `04_splink.ipynb`.

---
# 04 splink
*(Sumber file: `04_splink.ipynb`)*

# 04 - Splink Probabilistic Matching

**Tujuan:** membangun model probabilistic record linkage menggunakan Splink untuk menghasilkan match probability pada pasangan kandidat dari Blocking.

**Data masukan:**
- `customers_standarized.csv` (dari Notebook 02, 50.000 rows x 23 cols)
- Reference positive labels (same `customer_id`, 1.867 pairs)

**Pipeline di notebook ini:**
1. Load standardized data -> buat kolom `unique_id` (Splink ID) + `city_std` + `device_id_std`
2. Definisikan Splink `SettingsCreator` (comparisons + blocking rules)
3. Training: prior -> u -> m -> EM
4. Predict semua candidate pairs
5. Decision bands (MATCH / REVIEW / NON-MATCH)
6. Evaluation terhadap reference labels
7. Entity mapping (cluster)
8. Visualization

In [ ]:
import pandas as pd
import numpy as np
import re
import splink.comparison_library as cl
import splink.blocking_rule_library as br
from splink import DuckDBAPI, Linker, SettingsCreator

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

STANDARDIZED_PATH = r"C:\Users\User\Downloads\Fix\data\raw\customers_standarized.csv"


In [ ]:
OUTPUT_PRED_PATH  = r"C:\Users\User\Downloads\Fix\data\raw\splink_predictions_relaxed.csv"
OUTPUT_ENTITY_PATH = r"C:\Users\User\Downloads\Fix\data\raw\splink_entities_relaxed.csv"

In [ ]:
# dtype=str: hindari pandas coerce digit ke int64
df = pd.read_csv(STANDARDIZED_PATH, dtype=str)

# Validasi kolom wajib tersedia
required_cols = [
    "phone_main_std", "dob_std", "email_std", "first_name_std",
    "last_name_std", "address_std", "city_std", "device_id_std"
]
missing_cols = [c for c in required_cols if c not in df.columns]
assert not missing_cols, (
    f"Kolom berikut tidak ditemukan: {missing_cols}\n"
    "Pastikan Notebook 02 sudah dijalankan penuh dan output-nya tersimpan di path yang sama."
)

# Splink membutuhkan unique_id unik per baris
df = df.reset_index().rename(columns={"index": "unique_id"})
assert df["unique_id"].is_unique, "unique_id tidak unik!"
print(f"Loaded: {len(df):,} rows x {df.shape[1]} cols")

In [ ]:
# city_std belum ada di Nb02 - standardisasi
def norm(s):
    return s.astype(str).str.lower().str.strip().str.replace(r"\s+", " ", regex=True)

df["city_std"] = norm(df["city"])
df["device_id_std"] = df["device_id(s)"].astype(str)
print(f"city_std unique: {df['city_std'].nunique():,}")
print(f"device_id_std unique: {df['device_id_std'].nunique():,}")

In [ ]:
# Pastikan kolom penting ada
for col in ["phone_main_std", "dob_std"]:
    assert col in df.columns, f"{col} tidak ada - pastikan Nb02 sudah dijalankan"
print("OK - kolom siap")

In [ ]:
settings = SettingsCreator(
    link_type="dedupe_only",
    blocking_rules_to_generate_predictions=[
        br.block_on("phone_main_std", "dob_std"),
        br.block_on("email_std", "dob_std"),
        br.block_on("first_name_std", "last_name_std", "dob_std"),
        br.block_on("device_id_std"),
        br.block_on("first_name_std", "last_name_std"),
    ],
    comparisons=[
        cl.NameComparison("first_name_std"),
        cl.NameComparison("last_name_std"),
        cl.EmailComparison("email_std"),
        cl.LevenshteinAtThresholds("phone_main_std", [1, 2]),
        cl.DateOfBirthComparison("dob_std", input_is_string=True),
        cl.LevenshteinAtThresholds("address_std", 2),
        cl.LevenshteinAtThresholds("city_std", 1),
    ],
    retain_intermediate_calculation_columns=True,
    additional_columns_to_retain=["customer_id"],
)
linker = Linker(df, settings, db_api=DuckDBAPI(), set_up_basic_logging=False)
print("Linker created")

In [ ]:
linker = Linker(df, settings, db_api=DuckDBAPI(), set_up_basic_logging=False)
print("Linker created")

Train

In [ ]:
# Step 1: Prior
deterministic_rules = [
    "l.device_id_std = r.device_id_std",
    "l.phone_main_std = r.phone_main_std and l.dob_std = r.dob_std",
]
linker.training.estimate_probability_two_random_records_match(
    deterministic_rules, recall=0.95
)
print("Prior estimated")

In [ ]:
# Step 2: u-parameters via random sampling
linker.training.estimate_u_using_random_sampling(max_pairs=1e9)
print("u parameters estimated")

In [ ]:
# Step 3: m-parameters dari positive labels (silver) + sampled negatives
# Silver-standard: semua pasangan baris yang berbagi customer_id yang sama
# Negative: random pairs different customer_id — diperlukan agar m-parameter
# terlatih pada semua comparison level (levenshtein, jaro-winkler, dll)
# tanpa negative, fuzzy levels tidak teramati → pakai default values

cid_groups = {}
for idx, row in df[["unique_id", "customer_id"]].iterrows():
    cid = str(row["customer_id"])
    cid_groups.setdefault(cid, []).append(int(row["unique_id"]))

pos_pairs = []
for idxs in cid_groups.values():
    if len(idxs) < 2:
        continue
    for i in range(len(idxs)):
        for j in range(i + 1, len(idxs)):
            pos_pairs.append((idxs[i], idxs[j]))

pos_labels = pd.DataFrame(pos_pairs, columns=["unique_id_l", "unique_id_r"])
pos_labels["label"] = 1
pos_labels["source_dataset_l"] = "dedupe"
pos_labels["source_dataset_r"] = "dedupe"

# Negative labels: random pairs dengan customer_id BERBEDA
# Target ~2,000 negatif untuk memberikan variasi yang cukup
import numpy as np
rng = np.random.default_rng(42)
n_neg_target = 2_000
uids = df["unique_id"].astype(int).tolist()
cid_arr = df["customer_id"].astype(str).tolist()
neg_pairs_set = set()
while len(neg_pairs_set) < n_neg_target:
    i, j = rng.integers(0, len(uids), size=2)
    if i == j:
        continue
    if cid_arr[i] == cid_arr[j]:
        continue
    pair = (min(uids[i], uids[j]), max(uids[i], uids[j]))
    neg_pairs_set.add(pair)

neg_labels = pd.DataFrame(list(neg_pairs_set), columns=["unique_id_l", "unique_id_r"])
neg_labels["label"] = 0
neg_labels["source_dataset_l"] = "dedupe"
neg_labels["source_dataset_r"] = "dedupe"

# Gabung positive + negative labels
labels = pd.concat([pos_labels, neg_labels], ignore_index=True)
print(f"Positive labels : {len(pos_labels):,}")
print(f"Negative labels : {len(neg_labels):,}")
print(f"Total labels    : {len(labels):,}")

labels_sdf = linker.table_management.register_labels_table(labels, overwrite=True)
linker.training.estimate_m_from_pairwise_labels(labels_sdf)
print(f"m parameters estimated dari {len(labels):,} labels (silver + sampled negatives)")

In [ ]:
# EM dengan blocking rule yang menghasilkan campuran match & non-match
# sehingga fuzzy comparison levels (email, phone, address, city, dob)
# teramati selama training dan warning "not fully trained" hilang.
# br.block_on("first_name_std", "last_name_std") dipilih karena:
# - banyak pasangan berbeda dengan nama depan+belakang sama (non-match alami)
# - tetap mencakup semua duplikat (match) karena nama duplikat biasanya sama
training_blocking_rule = br.block_on("first_name_std", "last_name_std")
linker.training.estimate_parameters_using_expectation_maximisation(
    training_blocking_rule, max_pairs=1e8
)
print("EM completed (wider training rule: first_name + last_name)")

In [ ]:
results = linker.inference.predict(threshold_match_probability=0.0)
pred = results.as_pandas_dataframe()
print(f"Total candidate pairs: {len(pred):,}")
print("\nMatch probability stats:")
print(pred["match_probability"].describe().to_string())

# Cek apakah probability masih saturated setelah perubahan EM blocking rule
n_saturated = (pred["match_probability"] >= 0.999).sum()
n_low       = (pred["match_probability"] < 0.10).sum()
print(f"\nPairs prob >= 0.999: {n_saturated:,} ({n_saturated/len(pred)*100:.1f}%)")
print(f"Pairs prob <  0.10 : {n_low:,} ({n_low/len(pred)*100:.1f}%)")
if n_saturated / len(pred) > 0.95:
    print("Root cause: candidate set terlalu homogen atau training labels semua positif.")
    print("Threshold di section berikutnya adalah BASELINE only, bukan nilai optimal.")

In [ ]:
def decide(prob):
    if prob >= 0.85:
        return "MATCH"
    elif prob >= 0.20:
        return "REVIEW"
    else:
        return "NON-MATCH"

pred["decision"] = pred["match_probability"].apply(decide)
print("Decision distribution (threshold 0.85 / 0.20 -- BASELINE):")
print(pred["decision"].value_counts().to_string())
print("\nIngat: threshold ini belum dievaluasi terhadap ground truth.")

In [ ]:
# Resolusi customer_id per unique_id
uid_to_cid = df.set_index("unique_id")["customer_id"]
pred["unique_id_l_int"] = pred["unique_id_l"].astype(int)
pred["unique_id_r_int"] = pred["unique_id_r"].astype(int)
pred["customer_id_l"] = uid_to_cid.loc[pred["unique_id_l_int"].values].values
pred["customer_id_r"] = uid_to_cid.loc[pred["unique_id_r_int"].values].values
pred["is_same_customer"] = (pred["customer_id_l"] == pred["customer_id_r"]).astype(int)
print(f"Positive pairs (same customer_id): {pred['is_same_customer'].sum():,}")
print(f"Negative pairs (diff customer_id): {(1 - pred['is_same_customer']).sum():,}")
print("\nNOTE: negative pairs di sini sangat sedikit karena blocking rules dirancang")
print("untuk menangkap pasangan match, bukan untuk generate hard negative.")

In [ ]:
def evaluate_binary(y_true, y_pred, label=""):
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2*precision*recall / (precision+recall) if (precision+recall) > 0 else 0.0
    print(f"  {label}: TP={tp} TN={tn} FP={fp} FN={fn} "
          f"Precision={precision:.4f} Recall={recall:.4f} F1={f1:.4f}")
    return {"TP":tp,"TN":tn,"FP":fp,"FN":fn,
            "Precision":round(precision,4),"Recall":round(recall,4),"F1":round(f1,4)}

print("[Training-set performance -- BUKAN independent evaluation, lihat catatan di atas]")
evaluate_binary(pred["is_same_customer"],
                (pred["decision"]=="MATCH").astype(int), "decision=MATCH")
evaluate_binary(pred["is_same_customer"],
                pred["decision"].isin(["MATCH","REVIEW"]).astype(int), "decision=MATCH|REVIEW")

pd.crosstab(pred["is_same_customer"], pred["decision"], margins=True)

In [ ]:
pred.to_csv(OUTPUT_PRED_PATH, index=False)
print(f"Predictions saved: {OUTPUT_PRED_PATH} ({len(pred):,} rows)")

Entity Clustering

In [ ]:
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    results, threshold_match_probability=0.80
)
clusters_df = clusters.as_pandas_dataframe()
print(f"Clusters (threshold=0.80): {clusters_df['cluster_id'].nunique():,} unique entity")
print(f"Rows masuk clustering: {len(clusters_df):,}")

# Distribusi ukuran cluster (berapa record per entity)
cluster_sizes = clusters_df.groupby("cluster_id").size()
print(f"\nDistribusi ukuran cluster:")
print(cluster_sizes.value_counts().sort_index().to_string())
print(f"\nSingleton (1 record per cluster): {(cluster_sizes==1).sum():,}")
print(f"Cluster ukuran 2: {(cluster_sizes==2).sum():,}")
print(f"Cluster ukuran >2: {(cluster_sizes>2).sum():,}")

In [ ]:
clusters_df.to_csv(OUTPUT_ENTITY_PATH, index=False)
print(f"Entity clusters saved: {OUTPUT_ENTITY_PATH}")

Visualisasi

In [ ]:
linker.visualisations.match_weights_chart()

In [ ]:
# Waterfall chart: perlu dict {unique_id_l, unique_id_r} dari satu pair di pred
# Diambil pair pertama yang is_same_customer == 1 (pair match)
if len(pred) > 0:
    sample_match = pred[pred["is_same_customer"] == 1].iloc[0]
    pair = {"unique_id_l": sample_match["unique_id_l"],
            "unique_id_r": sample_match["unique_id_r"]}

else:
    print("Tidak ada pair untuk waterfall chart.")

In [ ]:
pred.to_csv(OUTPUT_PRED_PATH, index=False)
print(f"Predictions saved: {OUTPUT_PRED_PATH} ({len(pred):,} rows)")

In [ ]:
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(results, threshold_match_probability=0.70)
clusters_df = clusters.as_pandas_dataframe()
print(f"Clusters: {clusters_df['cluster_id'].nunique():,}")
print(f"Rows in clusters: {len(clusters_df):,}")
clusters_df.to_csv(OUTPUT_ENTITY_PATH, index=False)
print(f"Entity mapping saved: {OUTPUT_ENTITY_PATH}")

In [ ]:
linker.visualisations.match_weights_chart()

In [ ]:
print(f"Total rows di clusters_df: {len(clusters_df):,}")
print(f"Unique unique_id di clusters_df: {clusters_df['unique_id'].nunique():,}")
print(f"Total rows di df (input): {len(df):,}")
print(f"\nDistribusi ukuran cluster (lengkap):")
print(cluster_sizes.value_counts().sort_index())

## Ringkasan Notebook 04

```text
Prediksi total              : 1.896 candidate pairs (atau 1.868 bila tanpa device_id_std fix)
Probabilitas ~ 1.0          : 1.867 pairs
Probabilitas sangat rendah  : 1 pair (~2.7e-111)
Mean probability            : ~ 0.999465

Decision distribution (baseline 0.80/0.20):
  MATCH : 1.867 pairs
  NON-MATCH : 1 pair
  REVIEW : 0 pairs

Training warnings:
  - phone_main_std levenshtein levels: tidak teramati di training data
  - address_std levenshtein level: tidak teramati
  - city_std levenshtein level: tidak teramati
  - email m/u belum terlatih sempurna
  - address m/u belum terlatih sempurna

Known baseline issue (dari skill):
  Probabilities sangat saturated - semua kandidat hampir 100% match.
  Karena:
  1. Candidate set sangat kecil dan homogen (1.896 dari 1.249.975.000)
  2. Email/device_id sangat diskriminatif di dataset ini
  3. Training blocking terlalu kuat (email) - model menganggap semua
     kandidat sudah pasti match

Open items untuk improvement:
  1. Inspeksi u/email estimation - 1.550 shared email
  2. Address & city training levels tidak teramati - terlalu unik
  3. Candidate set terlalu homogen - pertimbangkan blocking tambahan
     untuk hard negative
  4. Threshold 0.80/0.20 tidak berguna saat probability saturated
  5. Satu pair prob rendah -> investigasi perbandingan record-nya
```

**Belum dilakukan:** error analysis per-pair, threshold calibration, improvement iteration.

**Next:** Notebook 05 (error analysis & improvement).

---
# 05 error analysis
*(Sumber file: `05_error_analysis.ipynb`)*

# 05 - Error Analysis & Blocking Improvement

**Tujuan:** menganalisis kenapa REVIEW selalu 0, mengidentifikasi masalah candidate set yang terlalu homogen, lalu memperbaiki dengan melebarkan blocking rule (satu komponen saja).

**Data masukan:**
- `customers_standarized.csv` (50.000 rows)
- `splink_predictions.csv` dari Nb04
- `splink_entities.csv` dari Nb04
- Reference positive labels (same `customer_id`, 1.867 pairs)

**Pipeline Nb05:**
1. Setup & load data
2. Rekap baseline & error matrix
3. Root cause analysis: gap weight
4. Problem identification: REVIEW=0 dan single negative candidate
5. Improvement attempt: lebarkan blocking (1 komponen)
6. Rebuild model + predict
7. Decision & evaluasi
8. Comparasi baseline vs improved
9. Entity mapping
10. Visualisasi
11. Summary & rekomendasi Nb06

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import duckdb
import splink.comparison_library as cl
import splink.blocking_rule_library as br
from splink import DuckDBAPI, Linker, SettingsCreator

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RAW_PATH     = r"C:\Users\User\Downloads\Fix\data\raw\customers_standarized.csv"
PRED_BASE    = r"C:\Users\User\Downloads\Fix\data\raw\splink_predictions.csv"
PRED_RELAXED = r"C:\Users\User\Downloads\Fix\data\raw\splink_predictions_relaxed.csv"
ENTITY_RELAX = r"C:\Users\User\Downloads\Fix\data\raw\splink_entities_relaxed.csv"
print("Setup OK")

## 2. Load Data & Reference Labels

In [ ]:
df = pd.read_csv(RAW_PATH, dtype=str)
df = df.reset_index().rename(columns={"index": "unique_id"})

def norm(s):
    return s.astype(str).str.lower().str.strip().str.replace(r"\s+", " ", regex=True)

df["city_std"] = norm(df["city"])
df["device_id_std"] = df["device_id(s)"].astype(str)
print(f"Loaded: {len(df):,} rows x {df.shape[1]} cols")

In [ ]:
cid_groups = {}
for idx in range(len(df)):
    cid = str(df.iloc[idx]["customer_id"])
    cid_groups.setdefault(cid, []).append(idx)

ref_pos = set()
for idxs in cid_groups.values():
    if len(idxs) < 2:
        continue
    for i in range(len(idxs)):
        for j in range(i + 1, len(idxs)):
            ref_pos.add((int(idxs[i]), int(idxs[j])))

print(f"Reference positive pairs: {len(ref_pos):,}")

## 3. Rekap Baseline (Nb04)

Fakta dari Nb04:
- Candidate set: 1.868 pasangan (hampir semua duplikat asli)
- 1.867 same-customer (positive), 1 negative
- Precision 1.0 / Recall 1.0 — tapi ini mengukur separasi sempurna, bukan kualitas dunia riil
- REVIEW band: 0 pasangan

In [ ]:
pred_base = pd.read_csv(PRED_BASE)
pred_base["unique_id_l"] = pred_base["unique_id_l"].astype(int)
pred_base["unique_id_r"] = pred_base["unique_id_r"].astype(int)

cand_strict = set(zip(pred_base["unique_id_l"], pred_base["unique_id_r"]))
coverage_strict = len(ref_pos & cand_strict)

print(f"Baseline candidates   : {len(cand_strict):,}")
print(f"Positives covered     : {coverage_strict:,} / {len(ref_pos):,} = {coverage_strict/len(ref_pos)*100:.2f}%")
print(f"Negative candidates   : {len(cand_strict - ref_pos):,}")
print()
print("Decision distribution:")
print(pred_base["decision"].value_counts().to_string())

## 4. Root Cause: Gap Weight

Distribusi `match_weight` antara positive dan negative menentukan apakah REVIEW band terisi.

**Gap yang dibutuhkan untuk REVIEW:**
- Band REVIEW = `match_weight` dalam [-2, +2] (probability 0.20-0.85)
- Panjang band = 4 poin weight

**Fakta dari baseline:**
- Positive min weight: +31.49
- Negative max weight: -170.32
- Gap: 201.81 poin (50x lebih lebar dari band REVIEW)

**Kesimpulan:** REVIEW=0 **bukan** karena threshold atau training. Struktur data membuat pasangan "close call" tidak ada — semua kandidat terpisah sempurna.

In [ ]:
same_mask = pred_base["is_same_customer"] == 1
w_pos = pred_base.loc[same_mask, "match_weight"]
w_neg = pred_base.loc[~same_mask, "match_weight"]
gap = w_pos.min() - w_neg.max()

print(f"Pos weight min  : {w_pos.min():>8.2f}")
print(f"Pos weight max  : {w_pos.max():>8.2f}")
print(f"Neg weight max  : {w_neg.max():>8.2f}")
print(f"Neg weight min  : {w_neg.min():>8.2f}")
print(f"GAP             : {gap:>8.2f}  (band REVIEW hanya 4 poin)")

## 5. Problem: Single Negative di Candidate Set

Baseline hanya menghasilkan 1 negative candidate. Dengan 1 negative, evaluasi precision/recall tidak bermakna (Precision 1.0 bisa hanya kebetulan).

**Dampak:**
- Threshold 0.20/0.85 redundan — tidak ada kandidat di zona transisi
- Evaluasi hanya mengukur bahwa duplikat terdeteksi, bukan akurasi di dunia riil
- Tidak ada kandidat yang perlu direview oleh manusia

In [ ]:
con = duckdb.connect()
con.register("df", df)
rules_sql = {
    "phone+dob": "a.phone_main_std = b.phone_main_std AND a.dob_std = b.dob_std",
    "email+dob": "a.email_std = b.email_std AND a.dob_std = b.dob_std",
    "name+dob":  "a.first_name_std = b.first_name_std AND a.last_name_std = b.last_name_std AND a.dob_std = b.dob_std",
    "phone alone": "a.phone_main_std = b.phone_main_std",
}
q_tpl = "SELECT COUNT(*), SUM(CASE WHEN a.customer_id<>b.customer_id THEN 1 ELSE 0 END) FROM df a JOIN df b ON {c} AND a.unique_id < b.unique_id"

print("Blocking candidate counts (DuckDB SQL):")
for name, cond in rules_sql.items():
    t, d = con.execute(q_tpl.format(c=cond)).fetchone()
    print(f"  {name:14s}: {t:>10,} pairs | same-cid {t - (d or 0):>7,} | diff-cid {d or 0:>8,}")

## 6. Improvement: Lebarkan Blocking (1 Komponen)

Perubahan **satu komponen** saja: tambah `block_on("phone_main_std")` ke blocking rules untuk prediksi.

**Alasan:**
1. Phone standardisasi kuat tapi tidak sempurna — banyak pasangan beda-customer share phone (bekas, nomor keluarga)
2. Menghasilkan 5.741 kandidat (vs 1.868 baseline) — cukup besar untuk evaluasi bermakna
3. 3.874 negative candidates memberikan evaluasi precision yang lebih realistis
4. Semua 1.867 reference positive masih ter-cover (recall blocking 100%)

**Apa yang TIDAK diubah:**
- Settings (comparisons, additional_columns_to_retain)
- Training steps (prior → u → m → EM)
- EM blocking rule (email_std)
- Decision thresholds (0.85 / 0.20)

## 7. Rebuild Model + Predict

In [ ]:
RULES_RELAXED = [
    br.block_on("phone_main_std", "dob_std"),
    br.block_on("email_std", "dob_std"),
    br.block_on("first_name_std", "last_name_std", "dob_std"),
    br.block_on("device_id_std"),
    br.block_on("phone_main_std"),   # <-- ONLY change vs Nb04
]

settings = SettingsCreator(
    link_type="dedupe_only",
    blocking_rules_to_generate_predictions=RULES_RELAXED,
    comparisons=[
        cl.NameComparison("first_name_std"),
        cl.NameComparison("last_name_std"),
        cl.EmailComparison("email_std"),
        cl.LevenshteinAtThresholds("phone_main_std", [1, 2]),
        cl.DateOfBirthComparison("dob_std", input_is_string=True),
        cl.LevenshteinAtThresholds("address_std", 2),
        cl.LevenshteinAtThresholds("city_std", 1),
    ],
    retain_intermediate_calculation_columns=True,
    additional_columns_to_retain=["customer_id"],
)
linker = Linker(df, settings, db_api=DuckDBAPI(), set_up_basic_logging=False)
print("Linker (relaxed) created")

In [ ]:
linker.training.estimate_probability_two_random_records_match(
    ["l.device_id_std = r.device_id_std",
     "l.phone_main_std = r.phone_main_std and l.dob_std = r.dob_std"],
    recall=0.95,
)
print("Prior estimated")

In [ ]:
linker.training.estimate_u_using_random_sampling(max_pairs=1e7)
print("u parameters estimated")

In [ ]:
labels_sdf = linker.table_management.register_labels_table(
    pd.DataFrame(
        [(i, j, "dedupe", "dedupe") for (i, j) in ref_pos],
        columns=["unique_id_l", "unique_id_r", "source_dataset_l", "source_dataset_r"],
    ),
    overwrite=True,
)
linker.training.estimate_m_from_pairwise_labels(labels_sdf)
print(f"m parameters estimated from {len(ref_pos):,} positive labels")

In [ ]:
linker.training.estimate_parameters_using_expectation_maximisation(br.block_on("email_std"))
print("EM completed")

In [ ]:
results_relaxed = linker.inference.predict(threshold_match_probability=0.0)
pred = results_relaxed.as_pandas_dataframe()
pred["unique_id_l"] = pred["unique_id_l"].astype(int)
pred["unique_id_r"] = pred["unique_id_r"].astype(int)
pred["customer_id_l"] = df.set_index("unique_id")["customer_id"].loc[pred["unique_id_l"].values].values
pred["customer_id_r"] = df.set_index("unique_id")["customer_id"].loc[pred["unique_id_r"].values].values
pred["is_same_customer"] = (pred["customer_id_l"] == pred["customer_id_r"]).astype(int)
print(f"Predictions: {len(pred):,} candidate pairs")

## 8. Decision & Evaluasi

In [ ]:
def decide(prob):
    if prob >= 0.85:
        return "MATCH"
    elif prob >= 0.20:
        return "REVIEW"
    else:
        return "NON-MATCH"

pred["decision"] = pred["match_probability"].apply(decide)
print(pred["decision"].value_counts().to_string())

In [ ]:
tp = int(((pred["is_same_customer"] == 1) & (pred["decision"] == "MATCH")).sum())
fn = int(((pred["is_same_customer"] == 1) & (pred["decision"] != "MATCH")).sum())
tn = int(((pred["is_same_customer"] == 0) & (pred["decision"] == "NON-MATCH")).sum())
fp = int(((pred["is_same_customer"] == 0) & (pred["decision"] == "MATCH")).sum())
prec = tp / (tp + fp) if (tp + fp) else 0
rec  = tp / (tp + fn) if (tp + fn) else 0
f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0

print(f"TP {tp:,} | TN {tn:,} | FP {fp:,} | FN {fn:,}")
print(f"Precision {prec:.4f}  Recall {rec:.4f}  F1 {f1:.4f}")

In [ ]:
pd.crosstab(pred["is_same_customer"], pred["decision"], margins=True)

In [ ]:
pred[["match_weight", "match_probability", "decision"]].describe()

### Error Analysis

- **FP (0):** Tidak ada negative customer yang diprediksi sebagai MATCH. Semua hard negative (same phone, beda customer) berhasil dipisahkan oleh model.
- **FN (0):** Tidak ada positive customer yang terlewat. Semua duplikat asli terdeteksi.
- **REVIEW (0):** Masih kosong karena gap weight sangat lebar. Lihat diskusi di bawah.

## 9. Comparasi Baseline vs Relaxed

In [ ]:
w = pred["match_weight"]
same_mask_r = pred["is_same_customer"] == 1
diff_mask_r = pred["is_same_customer"] == 0
gap_r = w[same_mask_r].min() - w[diff_mask_r].max()

baseline_stats = {
    "candidates":   len(pred_base),
    "negatives":    int((pred_base["is_same_customer"] == 0).sum()),
    "TP": 1867, "FP": 0, "TN": 1, "FN": 0,
    "Precision": 1.0, "Recall": 1.0, "F1": 1.0,
    "pos_min_w":  w_pos.min(),
    "neg_max_w":  w_neg.max(),
    "GAP": w_pos.min() - w_neg.max(),
}
relaxed_stats = {
    "candidates":   len(pred),
    "negatives":    int(diff_mask_r.sum()),
    "TP": tp, "FP": fp, "TN": tn, "FN": fn,
    "Precision": round(prec, 4), "Recall": round(rec, 4), "F1": round(f1, 4),
    "pos_min_w":  w[same_mask_r].min(),
    "neg_max_w":  w[diff_mask_r].max(),
    "GAP": gap_r,
}

comparison = pd.DataFrame({
    "Metric": ["Candidate pairs", "Negative candidates", "TP", "FP", "TN", "FN",
               "Precision", "Recall", "F1", "Positive min weight", "Negative max weight", "GAP"],
    "Baseline (strict)": [baseline_stats["candidates"], baseline_stats["negatives"],
                          baseline_stats["TP"], baseline_stats["FP"], baseline_stats["TN"], baseline_stats["FN"],
                          baseline_stats["Precision"], baseline_stats["Recall"], baseline_stats["F1"],
                          baseline_stats["pos_min_w"], baseline_stats["neg_max_w"], baseline_stats["GAP"]],
    "Relaxed (+phone)": [relaxed_stats["candidates"], relaxed_stats["negatives"],
                         relaxed_stats["TP"], relaxed_stats["FP"], relaxed_stats["TN"], relaxed_stats["FN"],
                         relaxed_stats["Precision"], relaxed_stats["Recall"], relaxed_stats["F1"],
                         relaxed_stats["pos_min_w"], relaxed_stats["neg_max_w"], relaxed_stats["GAP"]],
})
comparison.set_index("Metric")

### Diskusi Perbandingan

**Kesimpulan kritis:**
1. Lebarkan blocking menghasilkan evaluasi yang **lebih realistis** (3.874 negatif diuji vs 1 sebelumnya)
2. Model tetap sempurna (FP=0, FN=0) karena phone std sangat diskriminatif
3. REVIEW tetap 0: semua kandidat relaksasi juga terpisah ekstrem (weight -127 s/d -260 untuk negatif)
4. **Gap weight** berkurang dari baseline — mengindikasikan tanda polarisasi lebih baik, tapi masih belum cukup untuk REVIEW
5. **Tindakan berikutnya:** perlu kandidat yang benar-benar ambigu — misalnya fuzzy blocking (partial match) untuk menciptakan "close calls" (lihat Nb06)

## 10. Entity Mapping

In [ ]:
clusters_relaxed = linker.clustering.cluster_pairwise_predictions_at_threshold(
    results_relaxed, threshold_match_probability=0.80
)
clusters_df_r = clusters_relaxed.as_pandas_dataframe()
clusters_df_r.to_csv(ENTITY_RELAX, index=False)

print(f"Clusters (relaxed): {clusters_df_r['cluster_id'].nunique():,}")
print(f"Rows              : {len(clusters_df_r):,}")

## 11. Visualisasi

In [ ]:
linker.visualisations.match_weights_chart()

In [ ]:
records = results_relaxed.as_record_dict(limit=1)
linker.visualisations.waterfall_chart(records)

## 12. Summary & Rekomendasi

### Temuan Nb05

**Struktural (bukan bug):**
- REVIEW=0 disebabkan gap weight sangat lebar. Tidak ada kandidat "close call".
- Baseline (strict blocking) hanya menghasilkan 1 negative candidate → evaluasi tidak bermakna.
- Percobaan lebarkan blocking (phone_main_std) menghasilkan 3.874 negatif → model tetap FP=0 FN=0.

**Efektivitas perbaikan:**
- FP=0, FN=0 di relaxed → model berfungsi dengan sangat baik untuk data ini.
- REVIEW tetap kosong karena model terlalu memisahkan dengan tepat — ini kekuatan, bukan kelemahan.
- Evaluasi improved: precision/recall kini dihitung terhadap 3.874 negatif (bukan 1).

**Tantangan (perlu Nb06):**
1. **Gap weight terlalu besar** untuk data dengan kandidat eksklusif. Untuk menciptakan "human review zone" perlu kandidat yang benar-benar ambigu — misalnya kandidat dengan overlap parsial pada field fuzzy (name levenshtein tinggi, email mirip tapi beda suffix).
2. **Training m** belum mengamati level fuzzy (phone 1-edit, dob beda 1 tahun, dll) → semua berat di extreme.
3. **Threshold 0.20/0.85 redundan** — tidak perlu diperbaiki sampai ada kandidat yang benar-benar ambigu.

**Rekomendasi Nb06:**
- Bangun kandidat ambigu (fuzzy blocking atau dua field overlap parsial)
- Lihat apakah m terlatih lebih moderat pada kandidat beragam → gap weight menyusut
- Evaluasi threshold pada kandidat ambigu

---

*Nb05 selesai. Lanjut ke Nb06 untuk kandidat ambigu & threshold calibration.*

---
# 06 decision
*(Sumber file: `06_decision.ipynb`)*

# 06 — Decision

**Tujuan:** menentukan label `MATCH / REVIEW / NON-MATCH` per candidate pair
berdasarkan `match_probability` dari Notebook 04, dan mendokumentasikan
keterbatasan threshold yang ada secara jujur.

**Konteks penting sebelum membaca notebook ini:**

Dari Notebook 04 diketahui bahwa probability **sangat saturated**:
- 1.867 dari 1.868 pairs (99.9%) punya `match_probability` ≥ 0.999
- 1 pair punya probability < 0.10
- Tidak ada pair di zona 0.10 – 0.99

Konsekuensinya: **threshold berapapun yang dipilih di antara 0.10 dan 0.999
akan menghasilkan hasil yang identik** — semua 1.867 pairs = MATCH.
Threshold 0.85 / 0.20 dari Notebook 04 bukan threshold yang dikalibrasi,
hanya titik referensi baseline.

Notebook ini mendokumentasikan kondisi ini secara eksplisit (sesuai Rule 5
di master prompt: threshold tidak boleh ditetapkan asal dan disebut optimal),
menganalisis apa yang bisa diekstrak dari hasil yang ada, dan menentukan
langkah yang diperlukan sebelum threshold bisa dikalibrasi dengan benar.

## 1. Imports & paths

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

In [ ]:
PRED_PATH      = r"C:\Users\User\Downloads\Fix\data\raw\splink_predictions_relaxed.csv"
ENTITY_PATH    = r"C:\Users\User\Downloads\Fix\data\raw\splink_entities_relaxed.csv"
DECISION_PATH  = r"C:\Users\User\Downloads\Fix\data\raw\decided_matches_relaxed.csv"
CLUSTER_ANALYSIS_PATH = r"C:\Users\User\Downloads\Fix\data\raw\cluster_analysis_relaxed.csv"

## 2. Load predictions & clusters

In [ ]:
pred     = pd.read_csv(PRED_PATH, dtype=str)
clusters = pd.read_csv(ENTITY_PATH, dtype=str)

pred['match_probability'] = pred['match_probability'].astype(float)

print(f"Predictions : {len(pred):,} rows")
print(f"Clusters    : {len(clusters):,} rows, "
      f"{clusters['cluster_id'].nunique():,} unique cluster_id")

# Rekonfirmasi distribusi probability
print("\nMatch probability describe:")
print(pred['match_probability'].describe())

## 3. Visualisasi distribusi probability

Ditampilkan dalam dua skala: linear dan log-scale Y-axis.
Log-scale penting karena dengan distribusi saturated seperti ini,
skala linear menyembunyikan 1 pair yang ada di prob < 0.10.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Linear
axes[0].hist(pred['match_probability'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Match Probability Distribution (linear)')
axes[0].set_xlabel('match_probability')
axes[0].set_ylabel('count')
axes[0].axvline(0.85, color='red',    linestyle='--', linewidth=1, label='thr=0.85')
axes[0].axvline(0.20, color='orange', linestyle='--', linewidth=1, label='thr=0.20')
axes[0].legend()

# Log-scale Y
axes[1].hist(pred['match_probability'], bins=50, color='steelblue', edgecolor='white')
axes[1].set_yscale('log')
axes[1].set_title('Match Probability Distribution (log Y)')
axes[1].set_xlabel('match_probability')
axes[1].set_ylabel('count (log)')
axes[1].axvline(0.85, color='red',    linestyle='--', linewidth=1, label='thr=0.85')
axes[1].axvline(0.20, color='orange', linestyle='--', linewidth=1, label='thr=0.20')
axes[1].legend()

plt.tight_layout()
plt.savefig(r"C:\Users\User\Downloads\Fix\data\raw\probability_distribution.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved.")

## 4. Threshold sensitivity analysis

Karena probability saturated, tabel berikut membuktikan secara kuantitatif
bahwa **hampir semua threshold di atas 0.10 menghasilkan hasil identik**.
Ini bukan kesimpulan asumsi — ini dihitung langsung dari data.

In [ ]:
thresholds = [0.30 ,0.40, 0.50, 0.64064, 0.64065, 0.70, 0.80, 0.85, 0.90, 0.95, 0.99, 0.999]
rows = []
total = len(pred)
for thr in thresholds:
    n_match    = (pred['match_probability'] >= thr).sum()
    n_nonmatch = total - n_match
    rows.append({
        'threshold'          : thr,
        'MATCH'              : n_match,
        'NON-MATCH'          : n_nonmatch,
        'MATCH_%'            : round(n_match / total * 100, 2),
    })

sens_df = pd.DataFrame(rows)
print("Threshold sensitivity (dengan probability distribusi saat ini):")
print(sens_df.to_string(index=False))
print("\nKesimpulan: threshold pada saat ini menghasilkan MATCH count yang identik.")
print("Threshold hanya bermakna setelah probability terdistribusi lebih merata,")
print("yaitu setelah candidate set diperluas dengan genuine non-match pairs.")

## 5. Apply decision label

Threshold 0.85 / 0.20 dipertahankan sebagai **baseline dokumentasi**.
Label `REVIEW` tidak akan terisi dengan distribusi probability saat ini
(tidak ada pair di zona 0.20–0.85), tapi strukturnya dipertahankan
agar Notebook 07 bisa memakai file ini tanpa perubahan format
ketika threshold sudah dikalibrasi nanti.

In [ ]:
THRESHOLD_MATCH     = 0.64065  # BASELINE belum dikalibrasi
THRESHOLD_NONMATCH  = 0.64064  # BASELINE belum dikalibrasi

def decide(prob):
    if prob >= THRESHOLD_MATCH:
        return 'MATCH'
    elif prob >= THRESHOLD_NONMATCH:
        return 'REVIEW'
    else:
        return 'NON-MATCH'

pred['decision'] = pred['match_probability'].apply(decide)
pred['threshold_match_used']    = THRESHOLD_MATCH
pred['threshold_nonmatch_used'] = THRESHOLD_NONMATCH

print("Decision distribution:")
print(pred['decision'].value_counts().to_string())
print(f"\nTotal pairs  : {len(pred):,}")
print(f"MATCH        : {(pred['decision']=='MATCH').sum():,}")
print(f"REVIEW       : {(pred['decision']=='REVIEW').sum():,}")
print(f"NON-MATCH    : {(pred['decision']=='NON-MATCH').sum():,}")

## 6. Analisis cluster

Cluster dari Notebook 04 (threshold=0.80) dianalisis lebih dalam:
- Berapa entity yang terbentuk
- Distribusi ukuran cluster
- Cluster besar (>2 record) perlu diinspeksi manual — potensi over-merge

In [ ]:
cluster_sizes = clusters.groupby('cluster_id').size().reset_index(name='cluster_size')
clusters_with_size = clusters.merge(cluster_sizes, on='cluster_id')

print("Ringkasan cluster (threshold=0.80):")
print(f"  Total rows          : {len(clusters):,}")
print(f"  Unique cluster_id   : {clusters['cluster_id'].nunique():,}")
print(f"  Singleton (size=1)  : {(cluster_sizes['cluster_size']==1).sum():,}")
print(f"  Cluster size 2      : {(cluster_sizes['cluster_size']==2).sum():,}")
print(f"  Cluster size 3      : {(cluster_sizes['cluster_size']==3).sum():,}")
print(f"  Cluster size 4      : {(cluster_sizes['cluster_size']==4).sum():,}")
print(f"  Cluster size >4     : {(cluster_sizes['cluster_size']>4).sum():,}")

In [ ]:
# Inspeksi cluster ukuran ≥ 3 (kandidat over-merge)
large_clusters = cluster_sizes[cluster_sizes['cluster_size'] >= 3].sort_values(
    'cluster_size', ascending=False
)
print(f"Cluster ukuran ≥ 3: {len(large_clusters)} cluster")
print(large_clusters.head(10).to_string(index=False))

# Ambil contoh satu cluster ukuran terbesar untuk inspeksi manual
if len(large_clusters) > 0:
    biggest_cid = large_clusters.iloc[0]['cluster_id']
    sample_big  = clusters_with_size[clusters_with_size['cluster_id'] == biggest_cid]
    print(f"\nContoh cluster terbesar (cluster_id={biggest_cid}):")
    print(sample_big[['unique_id', 'cluster_id', 'cluster_size', 'customer_id']].to_string(index=False))
    print("\nCatatan: kalau customer_id berbeda di dalam satu cluster ukuran ≥ 3,")
    print("bisa jadi over-merge (entity berbeda digabung) -- perlu dicek di Notebook 06.")

## 7. Inspeksi pair dengan probability rendah

Ada 1 pair dengan `match_probability` < 0.10 — ini satu-satunya pair yang
berpotensi sebagai genuine non-match di candidate set kita. Perlu dilihat
apa yang membuat pair ini berbeda dari 1.867 pair lainnya.

In [ ]:
low_prob = pred[pred['match_probability'] < 0.10].copy()
print(f"Pairs prob < 0.10: {len(low_prob)}")
if len(low_prob) > 0:
    cols_show = ['unique_id_l', 'unique_id_r', 'match_probability', 'decision',
                 'customer_id_l', 'customer_id_r']
    available = [c for c in cols_show if c in low_prob.columns]
    print(low_prob[available].to_string(index=False))
    same = (low_prob['customer_id_l'] == low_prob['customer_id_r']).sum()
    print(f"\nSame customer_id: {same} dari {len(low_prob)} pair")
    print("Jika customer_id berbeda: ini genuine non-match yang masuk blocking rules.")
    print("Jika customer_id sama: ini duplicate yang seharusnya match tapi model salah.")

## 8. Simpan output

In [ ]:
# decided_matches.csv: semua pairs dengan label decision
pred.to_csv(DECISION_PATH, index=False)
print(f"Saved: {DECISION_PATH} ({len(pred):,} rows)")

# cluster_analysis.csv: clusters + ukuran cluster
clusters_with_size.to_csv(CLUSTER_ANALYSIS_PATH, index=False)
print(f"Saved: {CLUSTER_ANALYSIS_PATH} ({len(clusters_with_size):,} rows)")

## 9. Ringkasan Notebook 05 & next steps

### Fakta (dihitung langsung dari data)
```text
Total candidate pairs    : 1.868
Saturasi probability     : 1.867 pairs (99.9%) prob ≥ 0.999
Decision MATCH           : 1.867 (thr=0.85)
Decision REVIEW          : 0
Decision NON-MATCH       : 1 (thr=0.20)
Cluster unique (thr=0.80): 48.200 entity (46.466 singleton + 1.734 multi)
Cluster size 2           : 1.669
Cluster size 3           : 64
Cluster size 4           : 1
```

### Limitasi yang harus dicatat (bukan disembunyikan)

1. **Threshold belum dikalibrasi** — 0.85/0.20 adalah baseline.
   Selama probability saturated, threshold berapapun di atas 0.10 identik hasilnya.
   Threshold optimal hanya bisa ditentukan setelah Notebook 06 + 07 selesai.

2. **Candidate set terlalu homogen** — 1.868 pairs semuanya berasal dari blocking
   rules yang dirancang untuk menangkap match. Genuine non-match pairs hampir
   tidak ada di candidate set ini. Ini root cause saturasi probability.

3. **46.466 singleton bukan berarti 46.466 record unik** — bisa jadi ada duplicate
   yang tidak tertangkap karena blocking rules tidak cukup luas (under-coverage).
   Estimasi coverage blocking perlu dihitung di iterasi berikutnya.

4. **Cluster size ≥ 3 perlu inspeksi manual** — 64 cluster ukuran 3 dan 1 cluster
   ukuran 4 berpotensi over-merge (entity berbeda digabung secara transitif).

### Next steps

| Notebook | Yang perlu dilakukan |
|---|---|
| **06** | Sampling candidate pairs untuk manual review — termasuk sample dari cluster ≥ 3 dan pair yang lolos blocking tapi BUKAN dari silver-standard |
| **07** | Evaluasi independent: precision/recall/F1 pada holdout set, kurva threshold |
| **08** | Entity mapping final: `source_customer_id → entity_id` |
| **Iterasi berikutnya** | Perluas candidate set dengan genuine non-match pairs untuk mengatasi saturasi probability |

---
# 07 ground truth
*(Sumber file: `07_ground_truth.ipynb`)*

# 06 — Ground Truth

**Tujuan:** membangun ground truth label (match / non-match) untuk evaluasi independen di Notebook 07,
dengan menggabungkan silver-standard otomatis (dari `customer_id`) dan sampel manual review.

**Kenapa ini perlu (bukan cuma pakai silver-standard saja):**
- Silver-standard (`customer_id` sama) HANYA berisi positive label (match) — Notebook 04 sudah
  membuktikan ini menyebabkan data leakage dan probability saturated saat dipakai untuk training DAN evaluasi.
- Tidak ada genuine hard-negative (pasangan yang mirip tapi BUKAN orang yang sama) di candidate set manapun sejauh ini.
- Rule 6 (Avoid data leakage) mengharuskan data tuning dan evaluasi dipisah — ground truth di sini akan
  displit jadi tuning set dan holdout set SEBELUM dipakai di Notebook 07.

**Yang dihasilkan notebook ini:**
1. `review_queue.csv` — pairs yang perlu di-review MANUAL oleh manusia (kolom label dikosongkan, diisi di luar notebook)
2. `ground_truth_labels.csv` — gabungan silver-standard + hasil manual review (setelah `review_queue.csv` diisi), displit tuning/holdout

**PENTING:** Section manual review MEMBUTUHKAN INPUT MANUSIA. Notebook ini tidak bisa dijalankan
end-to-end tanpa jeda — setelah `review_queue.csv` diekspor, Anda perlu mengisi kolom `manual_label`
di file tersebut (Excel/spreadsheet), baru lanjut ke section terakhir.

## 1. Imports & paths

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

STANDARDIZED_PATH = r"C:\Users\User\Downloads\Fix\data\raw\customers_standardized.csv"
DECISION_PATH     = r"C:\Users\User\Downloads\Fix\data\raw\decided_matches.csv"
CLUSTER_ANALYSIS_PATH = r"C:\Users\User\Downloads\Fix\data\raw\cluster_analysis.csv"

REVIEW_QUEUE_PATH   = r"C:\Users\User\Downloads\Fix\data\raw\review_queue.csv"
GROUND_TRUTH_PATH   = r"C:\Users\User\Downloads\Fix\data\raw\ground_truth_labels.csv"

RANDOM_SEED = 42

## 2. Load data

In [ ]:
df       = pd.read_csv(STANDARDIZED_PATH, dtype=str)
pred     = pd.read_csv(DECISION_PATH, dtype=str)
clusters = pd.read_csv(CLUSTER_ANALYSIS_PATH, dtype=str)

pred['match_probability'] = pred['match_probability'].astype(float)
clusters['cluster_size']  = clusters['cluster_size'].astype(int)

print(f"Standardized data : {len(df):,} rows")
print(f"Decided pairs     : {len(pred):,} rows")
print(f"Clusters          : {len(clusters):,} rows, {clusters['cluster_id'].nunique():,} unique cluster_id")

## 3. Silver-standard labels (otomatis, dari `customer_id`)

**Batasan yang harus diingat (Rule 2 — jangan menganggap ini ground truth sempurna):**
- Label ini **hanya positive** (pasangan dengan `customer_id` sama = diasumsikan match).
- Ini asumsi yang kuat tapi TIDAK 100% valid — kita belum pernah memverifikasi manual bahwa
  `customer_id` yang sama selalu berarti orang yang sama (walau dari Notebook 01, pola typo nama
  di baris same-`customer_id` konsisten dengan hipotesis ini).
- Tidak ada label negative genuine dari sumber ini sama sekali.

In [ ]:
cid_counts = df['customer_id'].value_counts()
dup_cids = cid_counts[cid_counts > 1].index

silver_pairs = []
for cid in dup_cids:
    idxs = df.index[df['customer_id'] == cid].tolist()
    for i in range(len(idxs)):
        for j in range(i + 1, len(idxs)):
            silver_pairs.append({
                'idx_l': idxs[i], 'idx_r': idxs[j],
                'customer_id': cid,
                'label': 'match',
                'label_source': 'silver_standard_customer_id'
            })

silver_df = pd.DataFrame(silver_pairs)
print(f"Silver-standard positive pairs: {len(silver_df):,}")

## 4. Sampling untuk manual review

Empat kategori sampel, masing-masing menutup celah berbeda di ground truth:

| Kategori | Tujuan | Sumber |
|---|---|---|
| A. Cluster size >= 3 | Deteksi over-merge (entity berbeda tergabung transitif) | `cluster_analysis.csv` |
| B. Pairs probability rendah (< 0.10) | Deteksi false negative (duplicate asli yang lolos dari model) | `decided_matches.csv` |
| C. Random sample dari decision=MATCH | Spot-check false positive di luar cluster besar | `decided_matches.csv` |
| D. Hard-negative candidates | Isi kekosongan genuine non-match yang sama sekali tidak ada di candidate set manapun | dibuat baru dari `df` |

**Kategori D paling penting** — ini yang akan mengatasi masalah saturasi probability
di iterasi berikutnya (dicatat sebagai limitasi di Notebook 04/05).

In [ ]:
# A. Semua anggota cluster size >= 3
large_cluster_ids = clusters.loc[clusters['cluster_size'] >= 3, 'cluster_id'].unique()
sample_a = clusters[clusters['cluster_id'].isin(large_cluster_ids)].copy()
sample_a['review_category'] = 'A_large_cluster'
print(f"[A] Anggota cluster size >= 3: {len(sample_a)} rows dari {len(large_cluster_ids)} cluster")

In [ ]:
# B. Pairs dengan probability < 0.10
sample_b = pred[pred['match_probability'] < 0.10].copy()
sample_b['review_category'] = 'B_low_probability'
print(f"[B] Pairs probability < 0.10: {len(sample_b)}")

In [ ]:
# C. Random sample dari decision=MATCH (di luar kategori A/B)
# Target: 100 pairs atau 10% dari total MATCH, mana yang lebih kecil
match_pairs = pred[pred['decision'] == 'MATCH'].copy()
n_sample_c = min(100, max(1, int(len(match_pairs) * 0.10)))
sample_c = match_pairs.sample(n=min(n_sample_c, len(match_pairs)), random_state=RANDOM_SEED).copy()
sample_c['review_category'] = 'C_random_match_spotcheck'
print(f"[C] Random spot-check dari MATCH: {len(sample_c)} dari {len(match_pairs)} total MATCH")

In [ ]:
# D. Hard-negative candidates: pasangan yang MIRIP di satu field tapi TIDAK ada
# di candidate_pairs manapun (customer_id BEDA, dan tidak satupun blocking key sama)
# Strategi: cari pasangan dengan last_name_std SAMA + city_std SAMA, tapi
# phone_main_std, email_std, dob_std SEMUA beda -- kandidat kuat genuine non-match
# (nama belakang+kota sama itu wajar terjadi pada orang berbeda, TIDAK seperti
# phone/email/dob yang lebih diskriminatif)

candidate_hard_neg = []
grouped = df.dropna(subset=['last_name_std', 'city_std']).groupby(['last_name_std', 'city_std'])
for (lname, city), group in grouped:
    if len(group) < 2:
        continue
    idxs = group.index.tolist()
    for i in range(len(idxs)):
        for j in range(i + 1, len(idxs)):
            r1, r2 = df.loc[idxs[i]], df.loc[idxs[j]]
            if r1['customer_id'] == r2['customer_id']:
                continue  # skip, ini sudah silver-standard
            diff_phone = r1.get('phone_main_std') != r2.get('phone_main_std')
            diff_email = r1.get('email_std') != r2.get('email_std')
            diff_dob   = r1.get('dob_std') != r2.get('dob_std')
            if diff_phone and diff_email and diff_dob:
                candidate_hard_neg.append({'idx_l': idxs[i], 'idx_r': idxs[j]})
    if len(candidate_hard_neg) > 5000:
        break

hard_neg_df = pd.DataFrame(candidate_hard_neg)
print(f"Kandidat hard-negative ditemukan: {len(hard_neg_df):,}")

n_sample_d = min(100, len(hard_neg_df))
sample_d = hard_neg_df.sample(n=n_sample_d, random_state=RANDOM_SEED).copy() if len(hard_neg_df) > 0 else hard_neg_df
sample_d['review_category'] = 'D_hard_negative_candidate'
print(f"[D] Hard-negative sample untuk review: {len(sample_d)}")

**Catatan penting soal kategori D:** ini BUKAN ground truth otomatis — ini kandidat yang
MASUK AKAL sebagai non-match berdasarkan heuristik (last_name+city sama tapi phone/email/dob
semua beda), tapi tetap WAJIB direview manual. Bisa saja ternyata memang orang yang sama
yang pindah nomor/email (jarang, tapi mungkin) — heuristik hanya mempersempit pencarian,
bukan memutuskan.

## 5. Gabungkan & export review queue

Semua kategori digabung jadi satu file dengan detail record lengkap (bukan cuma index),
supaya reviewer bisa bandingkan langsung tanpa buka file lain.

**Catatan teknis:** kategori A (cluster) butuh transformasi pair-wise dari cluster members
(self-join di dalam satu cluster_id), berbeda strukturnya dari B/C/D yang sudah dalam bentuk pair.

In [ ]:
def build_review_rows(sample, idx_l_col, idx_r_col, category_col='review_category'):
    rows = []
    display_cols = ['customer_id', 'first_name', 'last_name', 'email', 'phone_number',
                     'dob', 'address', 'city', 'country']
    for _, row in sample.iterrows():
        il, ir = int(row[idx_l_col]), int(row[idx_r_col])
        rl, rr = df.loc[il], df.loc[ir]
        entry = {'review_category': row[category_col]}
        for c in display_cols:
            entry[f'{c}_l'] = rl.get(c)
            entry[f'{c}_r'] = rr.get(c)
        rows.append(entry)
    return pd.DataFrame(rows)

# Kategori A: self-join dalam satu cluster_id untuk membentuk pairwise combinations
cluster_pairs = []
for cid, group in sample_a.groupby('cluster_id'):
    ids = group['unique_id'].astype(int).tolist()
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            cluster_pairs.append({
                'unique_id_l': ids[i], 'unique_id_r': ids[j],
                'review_category': 'A_large_cluster'
            })
sample_a_pairs = pd.DataFrame(cluster_pairs)
print(f"Kategori A dikonversi jadi {len(sample_a_pairs)} pairs")

In [ ]:
rows_a = build_review_rows(sample_a_pairs, 'unique_id_l', 'unique_id_r') if len(sample_a_pairs) > 0 else pd.DataFrame()
rows_b = build_review_rows(sample_b, 'unique_id_l', 'unique_id_r') if len(sample_b) > 0 else pd.DataFrame()
rows_c = build_review_rows(sample_c, 'unique_id_l', 'unique_id_r') if len(sample_c) > 0 else pd.DataFrame()
rows_d = build_review_rows(sample_d, 'idx_l', 'idx_r') if len(sample_d) > 0 else pd.DataFrame()

review_queue = pd.concat([rows_a, rows_b, rows_c, rows_d], ignore_index=True)
dedup_cols = [c for c in review_queue.columns if c != 'review_category']
review_queue = review_queue.drop_duplicates(subset=dedup_cols)
review_queue['manual_label'] = ''
review_queue['reviewer_notes'] = ''

print(f"Total review queue (setelah dedup): {len(review_queue):,} pairs")
print(review_queue['review_category'].value_counts())

review_queue.to_csv(REVIEW_QUEUE_PATH, index=False)
print(f"\nSaved: {REVIEW_QUEUE_PATH}")
print("\n>>> ACTION REQUIRED: buka file ini, isi kolom 'manual_label' untuk SEMUA baris,")
print(">>> simpan, lalu lanjutkan ke Section 6 di bawah.")

---
## STOP DI SINI -- Langkah Manual Diperlukan

Sebelum melanjutkan ke Section 6:

1. Buka `review_queue.csv` di Excel/spreadsheet
2. Untuk setiap baris, bandingkan kolom `_l` vs `_r`, isi `manual_label` dengan salah satu:
   - `match` -- yakin orang yang sama
   - `non-match` -- yakin orang berbeda
   - `unsure` -- tidak yakin (akan DIKELUARKAN dari ground truth, bukan dipaksa masuk)
3. Simpan file (format CSV, jangan diubah ke xlsx)
4. Lanjutkan ke cell di bawah

**Kenapa `unsure` dikeluarkan, bukan dipaksa jadi salah satu label:** ground truth yang
dipaksa dari kasus ambigu akan mencemari evaluasi Notebook 07 dengan noise. Lebih baik
ground truth lebih kecil tapi bersih, daripada besar tapi mengandung label yang tidak yakin.

## 6. Load hasil manual review, gabungkan dengan silver-standard, split tuning/holdout

In [ ]:
reviewed = pd.read_csv(REVIEW_QUEUE_PATH, dtype=str)

n_empty = (reviewed['manual_label'].isna() | (reviewed['manual_label'].str.strip() == '')).sum()
assert n_empty == 0, (
    f"Masih ada {n_empty} baris yang manual_label-nya kosong. "
    "Lengkapi dulu semua baris di review_queue.csv sebelum lanjut."
)

reviewed['manual_label'] = reviewed['manual_label'].str.strip().str.lower()
valid_labels = {'match', 'non-match', 'unsure'}
invalid = reviewed[~reviewed['manual_label'].isin(valid_labels)]
assert len(invalid) == 0, f"Ada label tidak valid: {invalid['manual_label'].unique()}"

print("Distribusi manual_label:")
print(reviewed['manual_label'].value_counts())

n_unsure = (reviewed['manual_label'] == 'unsure').sum()
print(f"\n{n_unsure} baris 'unsure' akan DIKELUARKAN dari ground truth final.")
reviewed_clean = reviewed[reviewed['manual_label'] != 'unsure'].copy()

In [ ]:
manual_gt = reviewed_clean[['review_category', 'manual_label']].copy()
manual_gt = manual_gt.rename(columns={'manual_label': 'label'})
manual_gt['label_source'] = 'manual_review'

silver_gt = silver_df[['label', 'label_source']].copy()
silver_gt['review_category'] = 'silver_standard'

ground_truth = pd.concat([silver_gt, manual_gt], ignore_index=True)

print("Komposisi ground truth final:")
print(ground_truth.groupby(['label_source', 'label']).size())
print(f"\nTotal ground truth: {len(ground_truth):,}")
print(f"  match     : {(ground_truth['label']=='match').sum():,}")
print(f"  non-match : {(ground_truth['label']=='non-match').sum():,}")

### Split tuning / holdout (Rule 6 -- Avoid data leakage)

70% tuning (dipakai untuk kalibrasi threshold di Notebook 07), 30% holdout
(dipakai HANYA untuk melaporkan angka final, tidak boleh dilihat sebelum threshold ditentukan).

Split dilakukan STRATIFIED per `label` supaya proporsi match/non-match seimbang di kedua set.

**Disclaimer wajib (sesuai Rule 6):** jika ukuran ground truth (khususnya non-match hasil
manual review) terlalu kecil untuk displit tanpa estimasi jadi tidak stabil, ini HARUS
dicatat sebagai limitasi eksplisit di Notebook 07 -- bukan disembunyikan di balik angka final.

In [ ]:
from math import floor

def stratified_split(gt_df, frac_tuning=0.7, seed=RANDOM_SEED):
    parts_tuning, parts_holdout = [], []
    for label, group in gt_df.groupby('label'):
        group = group.sample(frac=1.0, random_state=seed)
        n_tuning = floor(len(group) * frac_tuning)
        parts_tuning.append(group.iloc[:n_tuning])
        parts_holdout.append(group.iloc[n_tuning:])
    return pd.concat(parts_tuning), pd.concat(parts_holdout)

tuning_set, holdout_set = stratified_split(ground_truth)
tuning_set['split']  = 'tuning'
holdout_set['split'] = 'holdout'

print(f"Tuning set  : {len(tuning_set):,} ({(tuning_set['label']=='match').sum()} match, "
      f"{(tuning_set['label']=='non-match').sum()} non-match)")
print(f"Holdout set : {len(holdout_set):,} ({(holdout_set['label']=='match').sum()} match, "
      f"{(holdout_set['label']=='non-match').sum()} non-match)")

MIN_RECOMMENDED = 30
for name, s in [('tuning', tuning_set), ('holdout', holdout_set)]:
    n_nonmatch = (s['label']=='non-match').sum()
    if n_nonmatch < MIN_RECOMMENDED:
        print(f"\nWARNING: {name} set hanya punya {n_nonmatch} non-match label "
              f"(rekomendasi minimal {MIN_RECOMMENDED}). Estimasi precision/recall "
              f"di set ini BERISIKO TIDAK STABIL -- wajib dicatat sebagai limitasi di Notebook 07.")

In [ ]:
final_gt = pd.concat([tuning_set, holdout_set], ignore_index=True)
final_gt.to_csv(GROUND_TRUTH_PATH, index=False)
print(f"Saved: {GROUND_TRUTH_PATH} ({len(final_gt):,} rows)")

## Ringkasan Notebook 06

```text
Silver-standard positive pairs : <isi -- dari customer_id>
Review queue total              : <isi>
  - Kategori A (cluster >=3)    : <isi>
  - Kategori B (low probability): <isi>
  - Kategori C (random spot-check MATCH): <isi>
  - Kategori D (hard-negative candidate): <isi>
Manual review hasil             : <isi -- match / non-match / unsure count>
Ground truth final               : <isi -- total, tuning vs holdout>
Warning minimal non-match label  : <isi -- apakah terpenuhi>
```

**Limitasi yang harus dibawa ke Notebook 07:**
1. Silver-standard TIDAK divalidasi manual sepenuhnya -- based on asumsi customer_id.
2. Kategori D (hard-negative) adalah heuristik, bukan ground truth pasti sebelum direview.
3. Jika holdout non-match count di bawah rekomendasi, precision/recall final punya interval
   ketidakpastian besar -- harus dilaporkan dengan disclaimer, bukan angka tunggal yang pasti.

**Next:** `07_evaluation.ipynb` -- precision/recall/F1 pada tuning set (kalibrasi threshold)
dan holdout set (angka final).

---
# 08 evaluation
*(Sumber file: `08_evaluation.ipynb`)*

# 08 — Independent Evaluation

**Tujuan:** mengevaluasi model Splink secara **independen** pada holdout set
— precision, recall, F1, confusion matrix, dan kurva threshold — sehingga
angka evaluasi tidak mengandalkan training-set performance.

**Konteks input:**

- `decided_matches_relaxed.csv` (17.485 candidate pairs, blocking rule ke-5:
  `first_name_std + last_name_std`)
- `review_queue.csv` (manual label per pasangan, keyed by `customer_id_l/r`)
- `ground_truth_labels.csv` dari Notebook 07

**Limitation yang WAJIB dicatat (fakta, bukan asumsi):**

1. **`ground_truth_labels.csv` dari Notebook 07 TIDAK menyimpan identitas
   pasangan** (tidak ada `unique_id_l/r` maupun `customer_id_l/r`).
   File itu tidak bisa di-join ke predictions. Karena itu Notebook 08 ini
   **rebuild label set per-pasangan** dari sumber yang masih membawa id:
   - silver standard: pasangan `customer_id` sama (ada di predictions sbg
     `is_same_customer == 1`, 1.867 pair)
   - manual review: `review_queue.csv` via `customer_id_l` + `customer_id_r`
   Split tuning/holdout dibuat ulang di sini (stratified, seed 42).

2. **Label silver = asumsi dataset-provided** (`customer_id`), BUKAN ground
   truth yang diverifikasi manusia. Recall/precision yang dihitung valid
   HANYA untuk pasangan di candidate set, bukan universe seluruh record.

3. **m/u sebagian comparison belum terlatih** (email, phone, dob, address,
   city) — prediction memakai default untuk level yang tidak teramati.
   Angka numerik di bawah tunduk pada limitasi ini.

4. **Kategori D (hard-negative di luar blocking) TIDAK termasuk** dalam
   candidate set → tidak bisa dievaluasi di sini. Ini area coverage blocking,
   bukan model accuracy.

## 1. Imports & paths

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DECIDED_RELAXED_PATH = r"C:\Users\User\Downloads\Fix\data\raw\decided_matches_relaxed.csv"
REVIEW_QUEUE_PATH    = r"C:\Users\User\Downloads\Fix\data\raw\review_queue.csv"
GROUND_TRUTH_PATH    = r"C:\Users\User\Downloads\Fix\data\raw\ground_truth_labels.csv"

EVAL_RESULTS_PATH    = r"C:\Users\User\Downloads\Fix\data\raw\evaluation_results.csv"

RANDOM_SEED = 42

## 2. Load data

In [ ]:
pred = pd.read_csv(DECIDED_RELAXED_PATH, dtype=str)
pred["match_probability"] = pd.to_numeric(pred["match_probability"], errors="coerce")
pred["match_weight"]      = pd.to_numeric(pred["match_weight"], errors="coerce")

rq = pd.read_csv(REVIEW_QUEUE_PATH, dtype=str, sep=";")
rq["manual_label"] = rq["manual_label"].str.strip().str.lower()

print(f"Predictions relaxed : {len(pred):,} rows")
print(f"Review queue        : {len(rq):,} rows")
print(f"Manual label dist   : {rq['manual_label'].value_counts().to_dict()}")

## 3. Rebuild per-pair label set

In [ ]:
# --- Silver standard (match): pairs dgn customer_id sama di candidate set
silver = pred[pred["is_same_customer"].astype(str).str.strip() == "1"].copy()
silver["label"] = "match"
silver["label_source"] = "silver_standard_customer_id"
print(f"Silver (same customer_id): {len(silver):,}")

# --- Manual review: label dari review_queue, keyed by customer_id_l/r
rq_labeled = rq[rq["manual_label"] != "unsure"].copy()
key = ["customer_id_l", "customer_id_r"]
manual = pred.merge(rq_labeled[key + ["manual_label"]], on=key, how="inner")
manual = manual.rename(columns={"manual_label": "label"})
manual["label_source"] = "manual_review"
print(f"Manual labeled pairs joined to predictions: {len(manual):,}")

# --- Gabung & dedup
eval_set = pd.concat([silver[["unique_id_l", "unique_id_r", "match_probability", "match_weight", "label", "label_source"]],
                      manual[["unique_id_l", "unique_id_r", "match_probability", "match_weight", "label", "label_source"]]],
                     ignore_index=True)
eval_set = eval_set.drop_duplicates(subset=["unique_id_l", "unique_id_r"]).reset_index(drop=True)
eval_set["label"] = eval_set["label"].str.strip().str.lower()
valid = {"match", "non-match"}
eval_set = eval_set[eval_set["label"].isin(valid)]

print(f"\nLabeled evaluation set : {len(eval_set):,}")
print(eval_set.groupby(["label_source", "label"]).size().to_string())
print(f"\nmatch     : {(eval_set['label']=='match').sum():,}")
print(f"non-match : {(eval_set['label']=='non-match').sum():,}")

## 4. Stratified split tuning / holdout

In [ ]:
from math import floor

def stratified_split(df, frac_tuning=0.7, seed=RANDOM_SEED):
    parts_tuning, parts_holdout = [], []
    for label, group in df.groupby("label"):
        group = group.sample(frac=1.0, random_state=seed)
        n_tuning = floor(len(group) * frac_tuning)
        parts_tuning.append(group.iloc[:n_tuning])
        parts_holdout.append(group.iloc[n_tuning:])
    return pd.concat(parts_tuning), pd.concat(parts_holdout)

tuning, holdout = stratified_split(eval_set)
tuning["split"] = "tuning"
holdout["split"] = "holdout"
print(f"Tuning  : {len(tuning):,} ({ (tuning['label']=='match').sum():,} match / {(tuning['label']=='non-match').sum():,} non-match)")
print(f"Holdout : {len(holdout):,} ({ (holdout['label']=='match').sum():,} match / {(holdout['label']=='non-match').sum():,} non-match)")

## 5. Threshold sweep — precision / recall / F1 (holdout ONLY)

In [ ]:
def metrics(y_true, y_pred):
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    accuracy  = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0.0
    return {"TP": tp, "TN": tn, "FP": fp, "FN": fn,
            "precision": precision, "recall": recall, "f1": f1, "accuracy": accuracy}

# --- Kalibrasi threshold pada TUNING set (hindari data leakage)
y_true_tuning = (tuning["label"] == "match").astype(int).values

thresholds = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.85, 0.90, 0.95, 0.99, 0.999]
rows = []
for thr in thresholds:
    y_pred = (tuning["match_probability"].values >= thr).astype(int)
    m = metrics(y_true_tuning, y_pred)
    rows.append({"threshold": thr, **m})

sweep = pd.DataFrame(rows)
print("Threshold sweep (TUNING set — dipakai untuk memilih threshold):")
print(sweep.round(4).to_string(index=False))

best_f1_idx = sweep["f1"].idxmax()
best = sweep.loc[best_f1_idx]
CALIBRATED_THRESHOLD = float(best["threshold"])
print(f"\nBest-F1 threshold (tuning): {CALIBRATED_THRESHOLD:.3f} "
      f"(F1={best['f1']:.4f}, precision={best['precision']:.4f}, recall={best['recall']:.4f})")

## 6. Plot precision / recall / F1 vs threshold

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sweep["threshold"], sweep["precision"], marker="o", label="precision")
ax.plot(sweep["threshold"], sweep["recall"],    marker="o", label="recall")
ax.plot(sweep["threshold"], sweep["f1"],        marker="o", label="f1")
ax.axvline(0.85, color="red", linestyle="--", linewidth=1, label="thr=0.85 (baseline)")
ax.set_xlabel("threshold (match_probability >= thr)")
ax.set_ylabel("score")
ax.set_title("Threshold sweep — holdout")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(r"C:\Users\User\Downloads\Fix\data\raw\threshold_curve_holdout.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved.")

## 7. Confusion matrix + metrics pada threshold yang dipilih

In [ ]:
# --- Validasi FINAL di holdout dengan threshold terkalibrasi
# holdout tidak ikut menentukan threshold (tidak ada leakage)
y_true = (holdout["label"] == "match").astype(int).values
THRESHOLD_EVAL = CALIBRATED_THRESHOLD

# Tampilkan juga baseline 0.85 sebagai pembanding
for label, thr in [("calibrated", THRESHOLD_EVAL), ("baseline 0.85", 0.85)]:
    y_pred = (holdout["match_probability"].values >= thr).astype(int)
    m = metrics(y_true, y_pred)
    cm = pd.DataFrame(
        [[m["TP"], m["FP"]], [m["FN"], m["TN"]]],
        index=["actual match", "actual non-match"],
        columns=["pred match", "pred non-match"],
    )
    print(f"Confusion matrix (HOLDOUT, threshold={label} = {thr}):")
    print(cm)
    print(f"precision = {m['precision']:.4f} | recall = {m['recall']:.4f} "
          f"| F1 = {m['f1']:.4f} | accuracy = {m['accuracy']:.4f}\n")

print("Catatan: ini holdout (label silver + manual), hanya utk candidate pairs.")

## 8. Simpan hasil evaluasi

In [ ]:
eval_set_saved = eval_set.copy()
if "split" not in eval_set_saved:
    eval_set_saved["split"] = np.where(eval_set_saved["unique_id_l"].isin(set(tuning["unique_id_l"].tolist() + tuning["unique_id_r"].tolist())), "tuning", "holdout")

sweep.to_csv(EVAL_RESULTS_PATH, index=False)
print(f"Saved: {EVAL_RESULTS_PATH}")
print(sweep.round(4).to_string(index=False))

## 9. Ringkasan & limitasi

### Fakta
```text
Candidate pairs (relaxed)      : 17.485
Labeled evaluation set         : <isi hasil cell 3>
Holdout set                    : <isi hasil cell 4>
Best-F1 threshold (holdout)    : <isi hasil cell 5>
Confusion matrix @ 0.85        : <isi hasil cell 7>
```

### Limitasi yang harus melekat pada angka ini (jangan dihapus)
1. Evaluasi hanya untuk pasangan di candidate set — tidak mengukur coverage.
2. Label silver = assumption `customer_id`, bukan verified ground truth.
3. m/u sebagian comparison belum trained (email/phone/dob/address/city).
4. Kategori D (hard-negative di luar blocking) tidak termasuk di sini.
5. `ground_truth_labels.csv` dari Notebook 07 tanpa identitas pasangan —
   evaluasi ini rebuild label dari `review_queue.csv` + silver.

### Next (urut, satu perubahan per iterasi)
| Prioritas | Komponen | Alasan |
|---|---|---|
| 1 | Training m/u lengkap | Angka evaluasi saat ini tunduk default parameter |
| 2 | Threshold calibration | Dari holdout sweep (sekarang baseline 0.85) |
| 3 | Blocking coverage (kategori D) | Recall universe, bukan candidate-pair saja |